# 📑 Hoist Monthly Report
 
## 🔗 Links
- **[BRD - Hoist Monthly Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/308379650/Updated+BRD+Hoist+Reporting+-+Line+by+Line+LbL)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-168dexpo)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-516)**  
  A list of findings from UAT.
 
---

## Initialization

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')
dbutils.widgets.text("mtd_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")
mtd_run = dbutils.widgets.get("mtd_run")


if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    if mtd_run == '1':
        run_date = spark.sql(f"SELECT MAX(BusinessDate) FROM `{env}-cdh`.derivative.dexposure WHERE BusinessDate <= '{now_date}'").collect()[0][0]
    else:
        run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

In [0]:
RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{run_date}'), CAST('{run_date}' AS DATE)) AS DATE)").collect()[0][0]

FirstWorkingDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

FirstCalendarDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

LastCalendarDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

PreviousMonthLastWorkingDate = spark.sql(f"""
    SELECT PreviousMonthLastWorkingDate
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' 
""").collect()[0][0]

SecondWorkingDateCurrentMonth = spark.sql(f"""
    SELECT CalendarDate
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
    AND BusinessDayOfMonth = 2 
""").collect()[0][0]

if mtd_run == '1': 
    SecondWorkingDateNextMonth = run_date
else: 
    SecondWorkingDateNextMonth = spark.sql(f"""
        SELECT CalendarDate
        FROM `{env}-cdh`.reference.ref_calendar 
        WHERE MONTH(PreviousMonthLastWorkingDate) = MONTH('{RefMonth}') AND YEAR(PreviousMonthLastWorkingDate) = YEAR('{RefMonth}') AND BusinessDayOfMonth = 2
    """).collect()[0][0]

print(f"run_date: {run_date}" )
print(f"RefMonth: {RefMonth}" )
print(f"FirstWorkingDate: {FirstWorkingDate}" )
print(f"FirstCalendarDate: {FirstCalendarDate}" )
print(f"LastCalendarDate: {LastCalendarDate}" )
print(f"PreviousMonthLastWorkingDate: {PreviousMonthLastWorkingDate}" )
print(f"RefMonth: {RefMonth}" )
print(f"SecondWorkingDateCurrentMonth: {SecondWorkingDateCurrentMonth}" )
print(f"SecondWorkingDateNextMonth: {SecondWorkingDateNextMonth}" )

## Loading Tables

In [0]:
migrationDate = spark.sql(f"""
                          SELECT min(cast(AssignmentDate as date)) AS migrationDate,
                          CASE WHEN portfolioName = 'Mercury' THEN '2019-07-01'
                               WHEN portfolioName = 'Orbit' THEN '2022-03-28'
                               WHEN portfolioName = 'Light' THEN '2022-11-07'
                               WHEN PortfolioName = 'Cell' THEN '2023-10-23'
                               WHEN PortfolioName = 'Pearl Portfolio A' THEN '2024-02-19' 
                               ELSE MIN(CAST(AssignmentDate AS DATE)) END AS cutoffdate,
                               prefix,
                          PortfolioName
                          FROM `{env}-cdh`.primary.portfolio
                          WHERE PortfolioName IN {PortfolioName} 
                          GROUP BY PortfolioName, prefix
                      """)

migrationDate.createOrReplaceTempView("migrationDate")

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

In [0]:
dexposure = spark.sql(f"""
      SELECT dexp.* EXCEPT (dexp.AssetSegment), 
            CASE WHEN dexp.AssetSegment IS NULL AND dexp.ClosedDate = md.MigrationDate THEN 'Closed at Migration Date' ELSE dexp.AssetSegment END AS AssetSegment,
            '{run_date}' AS RunDate
      FROM `{env}-cdh`.derivative.get_exposures_by_business_date('{run_date}', true)  dexp
      LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
      LEFT JOIN MigrationDate as md on dexp.PortfolioName = md.PortfolioName
      WHERE dexp.sourcesystem = 'profits' AND dexp.PortfolioName IN {PortfolioName}
      AND RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
                      """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
dexposure_all = spark.sql(f"""
    SELECT 
        ExposureKey, 
        CDI, 
        BusinessDate, 
        OutstandingTotalBalance, 
        IsActiveFlag
    FROM `{env}-cdh`.derivative.dexposure dexp
    LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
    WHERE sourcesystem = 'profits' 
    AND PortfolioName IN {PortfolioName}
    AND BusinessDate <= '{run_date}'
    AND RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
  """)

dexposure_all.createOrReplaceTempView("dexposure_all")

In [0]:
MaxContractBucket = spark.sql(f"""                      
SELECT AgreementKey
        , MAX(cast(Bucket as int)) as MaxBucket
FROM dexposure
WHERE SlaveFlag = False
GROUP BY AgreementKey
""")

MaxContractBucket.createOrReplaceTempView("MaxContractBucket")

## InitializationExposureInfo

In [0]:
initializationexposureinfo = spark.sql(f"""                      
select Exposurekey, InitialAssignmentBalance, 
CASE WHEN LEFT(Exposurekey,3) = 'MRC' THEN 'Mercury' 
WHEN LEFT(Exposurekey,3) = 'ORB' THEN 'Orbit' 
WHEN LEFT(Exposurekey,3) = 'CEL' THEN 'Cell' 
WHEN LEFT(Exposurekey,3) = 'PRA' THEN 'Pearl Portfolio A' 
WHEN LEFT(Exposurekey,3) = 'LGT' THEN 'Light' 
ELSE NULL END as PortfolioName
from `{env}-cdh`.reference.initializationexposureinfo
""")

initializationexposureinfo.createOrReplaceTempView("initializationexposureinfo")

In [0]:
ExposureRelations = spark.sql(f"""
    SELECT DISTINCT PortfolioName,
      regexp_replace(CDI, '[^0-9]', '') as CDI, 
      regexp_replace(ExposureKey, '[^0-9]', '') as ExposureKey, 
      PartyRelationshipType
    FROM `{env}-cdh`.primary.exposure
    WHERE sourcesystem = 'qca' 
    AND PortfolioName IN {PortfolioName} 
    AND BusinessDate <= '{run_date}'
""")
ExposureRelations.createOrReplaceTempView("ExposureRelations")

# dparty = spark.sql(f"""
#     select DISTINCT A.CDI, Segment 
#     from `{env}-cdh`.derivative.dparty A
#     WHERE BusinessDate = '{run_date}'
#     AND sourcesystem='profits' 
#     AND PortfolioName IN {PortfolioName}
#      """)
# dparty.createOrReplaceTempView("dparty")

dunderlaw_active = spark.sql(f"""
SELECT *
FROM (
    SELECT DISTINCT
        CDI,
        InLawFlag,
        UnderLawType,
        cast(SubmissionDate as date),
        ROW_NUMBER() OVER (
            PARTITION BY CDI
            ORDER BY
                CASE 
                  WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Βεβαίωση ευάλωτου' THEN 1
                  WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Πτώχευση' THEN 2
                  WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Εξυγίανση' THEN 3
                  WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Εξωδικαστικός μηχανισμός' THEN 4
                  WHEN UnderLawType = 'N3869'  THEN 5
                  WHEN UnderLawType = 'N4605/2019' THEN 6
                  ELSE 7
                END
        ) AS rn
    FROM `{env}-cdh`.derivative.dunderlaw
    WHERE BusinessDate = '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND ApplicationStatus NOT IN ('Rejected', 'Resigned')
    AND NOT (UnderLawType = 'N3869' AND SourceSystem = 'qca')     
    AND NOT (
      UnderLawType = 'N4738/2020' 
      AND LawSystem = 'Εξωδικαστικός μηχανισμός' 
      AND ApplicationStatus = '99 - Ολοκλήρωση αίτησης' 
      AND 
      ( cast(SUBSTRING(CompletionReason, 1, 3) as string) IN (
        '101', '103', '105', '108', '109', '113', '116', '117', '118', '119', '120', '121')
    ) OR CompletionReason = 'Εξωδικαστικός μηχανισμός - Ακύρωση από αιτούντα' 
    )

      AND InLawFlag = 1
) t
WHERE rn = 1; 
""")
dunderlaw_active.createOrReplaceTempView("dunderlaw_active")

## Settlementexposure

### Settlements Full

In [0]:
settlementsfull = spark.sql(
f'''
SELECT 
    *, 
    CASE WHEN 
            SettlementStatus IN ('IMPLEMENTED', 'PENDING FOR IMPLEMENTATION', 'Implemented','Partially Fulfilled','Pending Implementation', 'Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' ) 
            AND MaxBusinessDate = '{run_date}' THEN True 
         ELSE FALSE 
    END AS IsActiveSettlement 
FROM 
(
    SELECT  se.*, 
            se.BusinessDate as MaxBusinessDate, 
            RANK() OVER (PARTITION BY se.PortfolioName, 
                 case when left(se.Exposurekey,3) = p.Prefix then  se.Exposurekey else concat(p.Prefix,se.Exposurekey  ) end    
                 ORDER BY case 
                        when SettlementStatus IN ('IMPLEMENTED', 'PENDING FOR IMPLEMENTATION', 'Implemented','Partially Fulfilled','Pending Implementation', 'Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' )  then 1  -- active
                        when settlementstatus  IN ('FULFILLED', 'Fulfilled' ,'Partially Fulfilled' ,'Partially Fulfilled (Inactive SOR MGR)' ,'unknown' ) then 2  -- fulfilled
                        when settlementstatus  IN ('NOT FULFILLED', 'Cancelled' ,'Cancelled - Client Rejection' ,'Cancelled - Communication Failed' ,'Cancelled - Cosmos Migration' ,'Cancelled - GAIA Migration' ,'Cancelled - Galaxy Migration' ,'Cancelled - Internal Rejection' ,'Cancelled - Merged CDI' ,'Client Cancelled (Inactive SOR MGR)' ,'Internal Cancelled (Inactive SOR MGR)' ,'Non-Active (Inactive SOR MGR)' ,'Rejected (Inactive SOR MGR)' ,'For Approval (Inactive SOR MGR)' ,'Not Fulfilled' ,'Not Fulfilled (Inactive SOR MGR)' ,'Out of Collection'  ) then 3  -- inactive
                        else 4 end asc, 
                        ImplementationDate DESC, 
                        SubmissionDate DESC, 
                        CASE WHEN IsPrimarySettlementFlag = TRUE THEN 1 ELSE 0 END DESC, CDI DESC) AS Rank,
            case when left(se.Exposurekey,3) = p.Prefix then se.Exposurekey else concat(p.Prefix,se.Exposurekey) end as ExposureKeyPrefix
    FROM `p-cdh`.derivative.dsettlementexposure se  -- qa environment table non available
    left join `{env}-cdh`.primary.portfolio as p on p.BusinessDate = '{run_date}' and p.PortfolioName = se.PortfolioName 
    WHERE se.BusinessDate = '{run_date}'  
    AND se.PortfolioName IN {PortfolioName}
    AND se.SettlementType not in ('Άρση Εγγυητικής Σχέσης (SOR_MGR)', 'ΑΡΣΗ ΕΓΓΥΗΣΗΣ', 'Άρση Εγγυητικής Σχέσης')
    AND se.ExposureKey IS NOT NULL 
) 
WHERE Rank = 1

''')
settlementsfull.createOrReplaceTempView("settlementsfull")

### settlements agg

In [0]:
settlements_agg = spark.sql(     
f'''                    
select *
    , RANK() OVER (PARTITION BY PortfolioName,ExposureKeyPrefix  ORDER BY BusinessDate DESC) as rank
from (
    select
        se.PortfolioName, 
        se.BusinessDate,
        COUNT(DISTINCT se.SettlementKey) as cnt, 
        COUNT(CASE WHEN CAST(se.ImplementationDate AS DATE) BETWEEN md.cutoffdate AND CAST('{run_date}' AS DATE) THEN 1 ELSE NULL END) AS cnt2,
        COUNT(CASE WHEN CAST(se.ImplementationDate AS DATE) < md.cutoffdate THEN 1 ELSE NULL END) AS cnt3,
        case when left(se.Exposurekey,3) = p.Prefix then  se.Exposurekey else concat(p.Prefix,se.Exposurekey) end as ExposureKeyPrefix 
    FROM `p-cdh`.derivative.dsettlementexposure se  -- qa environment table non available
    LEFT JOIN MigrationDate md 
    ON md.PortfolioName = se.PortfolioName
    left join `{env}-cdh`.primary.portfolio as p on p.BusinessDate = '{run_date}' and p.PortfolioName = se.PortfolioName 
    WHERE se.BusinessDate <= '{run_date}'
    AND se.PortfolioName IN {PortfolioName}
    AND SettlementType not in ('Άρση Εγγυητικής Σχέσης (SOR_MGR)', 'ΑΡΣΗ ΕΓΓΥΗΣΗΣ') 
    GROUP BY 
          case when left(se.Exposurekey,3) = p.Prefix then  se.Exposurekey else concat(p.Prefix,se.Exposurekey  ) end, 
          se.PortfolioName, 
          se.BusinessDate
)
''')
settlements_agg.createOrReplaceTempView("settlements_agg")

### settlementexposure

In [0]:
settlementexposure = spark.sql(
f'''
SELECT
    de.PortfolioName,
    de.ExposureKey, 
    de.CDI,
    B.SettlementKey,
    B.CDI AS SettlementCDI, 
    de.DueAmount, 
    de.ExposureStatus,
    CASE WHEN B.ExposureKey IS NOT NULL THEN TRUE ELSE FALSE END AS HasActiveSettlementFlag,
    de.InitialAssignmentAmount, 
    B.SettlementType,
    B.SettlementStatus, 
    B.SubmissionDate,
    CAST(de.ClosedDate AS DATE) AS ClosedDate  , 
    B.IsMigratedFlag,
    B.ApprovalDate,
    B.KeptPercentage, 
    B.NumOfInstalments,
    B.FirstInstalmentDate,
    B.Bucket as SettlementBucket,
    CASE WHEN da.CDI IS NOT NULL THEN da.SubmissionDate ELSE B.SubmissionDate END AS AppliedDate,
    D.SettlementAmount,
    D.DiscountAmount,
    D.ProposedAmount, 
    D.PrincipalProposedAmount,
    D.RemainAmount,
    CASE WHEN B.ExposureKey IS NOT NULL AND C.cnt > 1 THEN 'Y' ELSE 'N' END AS HasMoreThan1ActiveSettlements,
    D.IsActiveSettlement, 
    CAST(D.ImplementationDate AS DATE) AS ImplementationDate, 
    C.cnt2 as CountOfImplementationsPerAccountAfterLTSA,
    C.cnt3 as CountOfImplementationsPerAccountBeforeLTSA
FROM dexposure de
LEFT JOIN settlementsfull B -- Active Settlements
ON  regexp_replace(B.ExposureKey, '[^0-9]', '') = regexp_replace(DE.ExposureKey, '[^0-9]', '') and 
    de.PortfolioName = B.PortfolioName and IsActiveSettlement = TRUE
LEFT JOIN settlements_agg c 
ON  regexp_replace(C.ExposureKeyPrefix, '[^0-9]', '') = regexp_replace(DE.ExposureKey, '[^0-9]', '') AND 
    de.PortfolioName = c.PortfolioName and c.rank = 1
LEFT JOIN settlementsfull D -- Active and Non-Active Settlements
ON  regexp_replace(D.ExposureKey, '[^0-9]', '') = regexp_replace(DE.ExposureKey, '[^0-9]', '') and 
    de.PortfolioName = D.PortfolioName 
LEFT JOIN dunderlaw_active da
ON de.CDI = da.CDI  

''')
settlementexposure.createOrReplaceTempView("settlementexposure")

## restructuring

In [0]:
restructuring = spark.sql(f"""
select DISTINCT 
    NewExposureKey, 
    RestructuringDate, 
    RestructuringType, 
    RestructuringAmount,
    RestructuringDiscountAmount, 
    BusinessDate, 
    PortfolioName, 
    rank
from (
    select DISTINCT 
        NewExposureKey, 
        RestructuringDate, 
        RestructuringType,
        RestructuringAmount, 
        RestructuringDiscountAmount, 
        BusinessDate,   
        PortfolioName, 
        rank() over (partition by NewExposureKey order by BusinessDate desc, case when RestructuringType IS NULL THEN 0 ELSE 1 END DESC) as Rank
    from  `{env}-cdh`.primary.restructuring 
    WHERE BusinessDate <= '{run_date}'
    AND PortfolioName IN {PortfolioName}
    and SourceSystem = 'profits'
) where rank = 1
""")
restructuring.createOrReplaceTempView("restructuring")

## paymentschedule 

In [0]:
paymentschedule = spark.sql(f"""
    SELECT *
    FROM `{env}-cdh`.primary.paymentschedule 
    WHERE BusinessDate = '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND PaymentScheduleType = 'Settlement Payment Schedule'
    
""")
paymentschedule.createOrReplaceTempView("paymentschedule")

## dpaymentschedule

In [0]:
dpaymentschedule = spark.sql(f"""
    SELECT *
    FROM `{env}-cdh`.derivative.dpaymentschedule 
    WHERE BusinessDate = '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND PaymentScheduleType = 'Loan Payment Schedule'
""")
dpaymentschedule.createOrReplaceTempView("dpaymentschedule")

## dportfoliotoolthirdparties

In [0]:
dportfoliotoolthirdparties = spark.sql(f"""
    select DISTINCT *
    from `{env}-cdh`.derivativeactive.dportfoliotoolthirdpartiesmonthly -- TO REVIEW FOR MtD 
    WHERE BusinessDate = '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND frequencytype='Monthly'
 """) 
dportfoliotoolthirdparties.createOrReplaceTempView("dportfoliotoolthirdparties")

## casecode

In [0]:
casecode = spark.sql(
f'''
    select CaseCodeKey, StrategyStep, StrategyType, TRIM(
        REPLACE(
            REPLACE(
                REPLACE(
                    REPLACE(
                        REPLACE(
                            Queue,
                            'Mercury', ''),
                        'Cell', ''),
                    'Orbit', ''),
                'Light', ''),
            'Pearl', '')
    ) AS QueueTrimmed
    from `{env}-cdh`.primary.casecode 
    where BusinessDate = '{run_date}'
    AND PortfolioName IN {PortfolioName}
''')
casecode.createOrReplaceTempView("casecode")

## Party

In [0]:
party = spark.sql(f"""    
      select DISTINCT A.PortfolioName, a.CDI, A.Segment, A.subsegment, A.ExtraSegment 
      from `{env}-cdh`.primary.party as A
      INNER JOIN (
            select CDI, max(businessdate) as BusinessDate
            from `{env}-cdh`.primary.party
            where BusinessDate <= '{run_date}'
            AND PortfolioName IN {PortfolioName}
            and sourcesystem in ('qc','qca')  
            group by CDI) as B ON A.CDI = B.CDI AND A.BusinessDate = B.BusinessDate
      where PortfolioName IN {PortfolioName}
      and sourcesystem in ('qc','qca')  
""")
party.createOrReplaceTempView("party")

## External Files

In [0]:
LetterFlag = spark.sql(f"""
    select DISTINCT cast(AFM as string)
    from `{env}-ReportInputFiles`.investor.mercury_letterflag
    """)
LetterFlag.createOrReplaceTempView("LetterFlag")

## AutoLoan

In [0]:
product = spark.sql(
f'''
select distinct 
    ProductKey
    , ProductDescription 
from `{env}-cdh`.primary.product
where BusinessDate = '{run_date}'
and sourcesystem in ('profits', 'qca')  
and (   --Product    
        LOWER(ProductDescription) like '%αυτο%' 
        or LOWER(ProductDescription) like '%auto%'
        or LOWER(ProductDescription) like '%moto%'
        -- SubProduct
        or LOWER(SubProduct) like '%αυτ/%'
        or LOWER(SubProduct) like '%αυτοκ/%'
        or LOWER(SubProduct) like '%αυτο%' 
        or LOWER(SubProduct) like '%auto%'
        or LOWER(SubProduct) like '%moto%'
    )
''')
product.createOrReplaceTempView("product")

## Collateral fields

In [0]:
dcollateral = spark.sql(f"""
    select * 
    from `{env}-cdh`.derivative.dcollateral
    where BusinessDate = '{run_date}' and PortfolioName IN {PortfolioName}
  """)
dcollateral.createOrReplaceTempView("dcollateral")

realestateinfo = spark.sql(f"""
    select * 
    from `{env}-cdh`.primary.realestateinfo
    where BusinessDate = '{run_date}' and SourceSystem = 'thesis'
  """)
realestateinfo.createOrReplaceTempView("realestateinfo")

drealestate = spark.sql(f"""
    select *
    from `{env}-cdh`.derivative.drealestate
    where BusinessDate = '{run_date}'
  """)
drealestate.createOrReplaceTempView("drealestate")

In [0]:
CollateralMapping = spark.sql(f"""
  SELECT DISTINCT
      dcol.BusinessDate,
      dcol.PortfolioName,
      dre.RealEstateCepalKey,
      re.OwnershipKey,
      dex.ExposureKey,
      dcol.AgreementKey,
      dcol.DCollateralKey,
      dcol.CollateralAmount,
      dcol.CollateralType,
      dcol.InitialLegalOrder,
      dcol.RealEstateCollateralStatus,
      case when dcol.CollateralType ='RE' then (
      case when dcol.subcategory = 'Προσημείωση' THEN 'Prenotation' 
      else 'Mortgage' end) 
      else Null END AS LienType, 
      dex.CDI,
      dcol.Currency
  FROM dcollateral dcol
  LEFT JOIN realestateinfo re
  ON dcol.RealEstateKey = re.RealEstateKey
  AND re.OwnerPartyKey = dcol.OwnerPartyKey
  LEFT JOIN dexposure dex
  ON dex.AgreementKey = dcol.AgreementKey
  LEFT JOIN drealestate dre
  ON dcol.RealEstateKey = dre.RealEstateKey
  WHERE dcol.CollateralType = 'RE'
  and dcol.CollateralStatus = 'Active'
""")
CollateralMapping.createOrReplaceTempView('CollateralMapping')

In [0]:
RankedAgreements = spark.sql(f"""
    SELECT 
        ExposureKey, 
        AgreementKey,
        DCollateralKey,
        InitialLegalOrder,
        SUM(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey) AS CollateralTotalAmount,
        SUM(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey, InitialLegalOrder) AS CollateralLienTotalAmount,
        ROW_NUMBER() OVER (PARTITION BY ExposureKey, AgreementKey ORDER BY InitialLegalOrder ASC, DCollateralKEY ASC) AS Rank,
        MIN(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey) AS MinCollateralTotalAmount
    FROM CollateralMapping
""")
RankedAgreements.createOrReplaceTempView("RankedAgreements")

CollateralMappingTempLien = spark.sql(f"""
    SELECT ExposureKey, 
        AgreementKey, 
        DCollateralKey,
        InitialLegalOrder AS MinCollateralLien,
        CollateralTotalAmount,
        CollateralLienTotalAmount AS MinCollateralLienTotalAmount
    FROM RankedAgreements
    WHERE Rank = 1
""")
CollateralMappingTempLien.createOrReplaceTempView("CollateralMappingTempLien")

## Balance Range

In [0]:
CDITaxNumberDistinct = spark.sql(f"""
    SELECT DISTINCT PortfolioName, CDI,PartyTaxRegistrationNumber
    FROM dexposure
    WHERE PartyTaxRegistrationNumber IS NOT NULL 
    and PartyTaxRegistrationNumber  NOT LIKE '0000000%' 
    and PartyTaxRegistrationNumber  NOT LIKE '9999999%'
""")
CDITaxNumberDistinct.createOrReplaceTempView("CDITaxNumberDistinct")

CountTaxNumberPerCDI = spark.sql(f"""
    SELECT PortfolioName, CDI
    FROM CDITaxNumberDistinct
    GROUP BY PortfolioName, CDI
    HAVING count(distinct PartyTaxRegistrationNumber) > 1
""")
CountTaxNumberPerCDI.createOrReplaceTempView("CountTaxNumberPerCDI")

CDITaxNumber = spark.sql(f"""
    SELECT DISTINCT DE.PortfolioName, DE.CDI,
        CASE WHEN A.PartyTaxRegistrationNumber IS NOT NULL AND E.CDI IS NULL THEN A.PartyTaxRegistrationNumber ELSE DE.CDI END AS CDITaxNumber,
        CASE
            WHEN DE.IsActiveFlag = false THEN 0
            ELSE DE.OutstandingTotalBalance
        END AS OutstandingTotalBalance,
        DE.InitialAssignmentAmount
    FROM dexposure AS DE
    LEFT JOIN CDITaxNumberDistinct AS A ON A.CDI = DE.CDI AND A.PortfolioName = DE.PortfolioName
    LEFT JOIN CountTaxNumberPerCDI AS e ON E.CDI = DE.CDI AND E.PortfolioName = DE.PortfolioName
""")
CDITaxNumber.createOrReplaceTempView("CDITaxNumber")

TaxBalance = spark.sql(f"""
    SELECT PortfolioName, 
            CDITaxNumber, 
            SUM(OutstandingTotalBalance) AS OutstandingTotalBalance, 
            SUM(InitialAssignmentAmount) AS InitialAssignmentAmount
    FROM CDITaxNumber
    GROUP BY PortfolioName, CDITaxNumber
""")
TaxBalance.createOrReplaceTempView("TaxBalance")

BalanceRange = spark.sql(f"""
    SELECT CDITaxNumber,PortfolioName,  
        CASE
        WHEN OutstandingTotalBalance < 5000 THEN CONCAT(PortfolioName, ' < 5k')
        WHEN (OutstandingTotalBalance >= 5000 AND OutstandingTotalBalance <= 10000) THEN CONCAT(PortfolioName, ' 5k - 10k')
        WHEN (OutstandingTotalBalance > 10000 AND OutstandingTotalBalance <= 20000) THEN CONCAT(PortfolioName, ' 10k - 20k')
        WHEN (OutstandingTotalBalance > 20000 AND OutstandingTotalBalance <= 50000) THEN CONCAT(PortfolioName, ' 20k - 50k')
        WHEN (OutstandingTotalBalance > 50000 AND OutstandingTotalBalance <= 75000) THEN CONCAT(PortfolioName, ' 50k - 75k')
        WHEN (OutstandingTotalBalance > 75000 AND OutstandingTotalBalance <= 125000) THEN CONCAT(PortfolioName, ' 75k - 125k')
        WHEN (OutstandingTotalBalance > 125000 AND OutstandingTotalBalance <= 200000) THEN CONCAT(PortfolioName, ' 125k - 200k')
        WHEN OutstandingTotalBalance > 200000 THEN CONCAT(PortfolioName, ' 200k+') END as BalanceRange
    FROM TaxBalance
""")
BalanceRange.createOrReplaceTempView("BalanceRange")  

TaxBalanceRange = spark.sql(f"""
    SELECT DISTINCT A.CDI, B.BalanceRange
    FROM CDITaxNumber A
    INNER JOIN BalanceRange B
    ON A.CDITaxNumber = B.CDITaxNumber
       AND A.PortfolioName = B.PortfolioName
""")
TaxBalanceRange.createOrReplaceTempView("TaxBalanceRange")

## Rejected Offers fields

In [0]:
rej_off = spark.sql(
f'''
select * from (
  select
   AgreementKey,
   HaircutAmount as AmountOfHaircutLastRejectedOffer,
   HaircutPercentage as HaircutPercentOfGBVLastRejectedOffer,
   HaircutPercentageOnBalance as HaircutPercentPrincipalBalanceLastRejectedOffer,
   TotalTenureInMonths as CountOfInstallmentsLastRejectedOffer,
   InstallmentAmount as MonthlyInstallmentAmountLastRejectedOffer,
   BusinessDate,
   CreatedDate,
   row_number() over (partition by AgreementKey order by CreatedDate desc) as rn       
  from `{env}-cdh`.primary.proposedrestructuring
  where FinalRestructuringStatus = 'Debtor Disagreed'
  and BusinessDate <= '{run_date}'
  AND PortfolioName IN {PortfolioName}
) where rn = 1
''')
rej_off.createOrReplaceTempView("rej_off")

### Installments

In [0]:
installments_v0 = spark.sql(f"""
SELECT 
    SE.Exposurekey, 
    COUNT(DISTINCT CASE WHEN coalesce(A.InstallmentStatus, B.InstallmentStatus) = 'Overdue' THEN coalesce(A.InstallmentNumber, B.InstallmentNumber) ELSE NULL END) AS OverdueInstallments,
    SUM(CASE WHEN Cast(coalesce(A.InstallmentDate, B.InstallmentDate) as DATE) >= {FirstCalendarDate!r} AND Cast(coalesce(A.InstallmentDate, B.InstallmentDate) as DATE) <= {LastCalendarDate!r} THEN abs(coalesce(A.InstallmentAmount, B.InstallmentAmount)) ELSE NULL END) AS MonthlyInstallmentAmount,
    COUNT(CASE WHEN coalesce(A.InstallmentDate, B.InstallmentDate) <= '{run_date}' AND coalesce(A.InstallmentAmountPaid, B.InstallmentAmountPaid) > 0 THEN  coalesce(A.InstallmentNumber, B.InstallmentNumber) ELSE NULL END) AS PastInstallments, 
    MAX(coalesce(A.InstallmentNumber, B.InstallmentNumber)) as NoOfInstallments, 
    MIN(Cast(CASE WHEN coalesce(A.InstallmentNumber, B.InstallmentNumber) = 1 THEN coalesce(A.InstallmentDate, B.InstallmentDate) ELSE NULL END as DATE)) as FirstInstallmentDate, 
    MIN(CASE WHEN coalesce(A.InstallmentDate, B.InstallmentDate) >= se.AppliedDate and se.IsActiveSettlement = TRUE And SE.ImplementationDate is null THEN coalesce(A.InstallmentDate, B.InstallmentDate) ELSE NULL END) as FirstInstallmentDateAfterApplication,
    MAX(CASE WHEN  coalesce(A.InstallmentNumber, B.InstallmentNumber) = 1 AND coalesce(A.DownPaymentFlag, B.DownPaymentFlag) is True and se.AppliedDate is not null THEN True ELSE FALSE END) AS DownPaymentFlagForApplied,
    SUM(A.InstallmentAmount) as ScheduledAmount
FROM SettlementExposure AS SE
LEFT JOIN paymentschedule AS A
ON A.SettlementKey = SE.SettlementKey 
LEFT JOIN dpaymentschedule AS B
ON B.ExposureKey = SE.ExposureKey
GROUP BY SE.ExposureKey
""")
installments_v0.createOrReplaceTempView("installments_v0")

### Final Settlement Table

In [0]:
settlement_fields = spark.sql(f"""

SELECT SE.*, 
    CASE 
        WHEN LatestImplementation = 'Settlement' AND IsActiveSettlement = TRUE THEN SE.SettlementAmount 
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' THEN SE.InitialAssignmentAmount 
    END AS SettlementAmount2, 
    CASE 
        WHEN LatestImplementation = 'Settlement' THEN CAST(SE.DiscountAmount AS FLOAT) 
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' THEN SE.RestructuringDiscountAmount 
    END AS HaircutAmount,
    CASE 
        WHEN LatestImplementation = 'Settlement' THEN 1.00 - (CASE WHEN COALESCE(CAST(SE.ProposedAmount AS FLOAT),0)= 0 THEN CAST(0 AS FLOAT)   ELSE (SE.SettlementAmount/SE.ProposedAmount) END)
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' THEN 
            CASE 
            WHEN COALESCE(CAST(SE.RestructuringAmount AS FLOAT),0) = CAST(0 AS FLOAT) THEN CAST(0 AS FLOAT) 
            ELSE  CAST(SE.RestructuringDiscountAmount AS FLOAT)/CAST(SE.RestructuringAmount AS FLOAT) 
            END
    END AS HaircutPercentage,
    CASE 
        WHEN LatestImplementation = 'Settlement' THEN 1.00 - (CASE WHEN COALESCE(CAST(SE.PrincipalProposedAmount AS FLOAT),0)=0 THEN CAST(0 AS  FLOAT) ELSE (SE.SettlementAmount/SE.PrincipalProposedAmount) END)
    END AS HaircutPrincipalPercentage,
    CASE 
        WHEN LatestImplementation = 'Settlement' THEN IFNULL(SE.RemainAmount,0)
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' THEN IFNULL(SE.RestructuringAmount,0) - IFNULL(SE.RestructuringDiscountAmount,0)
    END AS TotalAmountDueAfterHaircut,
    CASE 
        WHEN LatestImplementation = 'Settlement' AND IsActiveSettlement = TRUE THEN SE.SettlementStatus
        WHEN LatestImplementation = 'Restructuring' THEN 'Restructure'
    ELSE 'No Settlement/Restructure' END AS Status,
    CASE 
        WHEN LatestImplementation = 'Settlement' AND IsActiveSettlement = TRUE AND SE.NumOfinstalments <= 6 THEN 'Short Term (6M)'
        WHEN LatestImplementation = 'Settlement' AND IsActiveSettlement = TRUE AND SE.NumOfinstalments > 6 THEN 'Long Term (+6M)'
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' AND SE.NumberOfInstallments <= 6 THEN 'Short Term (6M)'
        WHEN LatestImplementation = 'Restructuring' AND SE.ExposureStatus <> 'Writen Off' AND SE.NumberOfInstallments > 6 THEN 'Long Term (+6M)'
    END AS ShortTerm_LongTermFlag,
    CASE 
        WHEN LatestImplementation = 'Settlement' AND IsActiveSettlement = TRUE THEN coalesce(SE.FirstInstalmentDate, se.ImplementationDate)
        WHEN LatestImplementation = 'Restructuring' THEN FirstInstallmentDate 
    END AS FirstInstallmentDate2,
    CASE WHEN B.CDI IS NOT NULL THEN UnderLawType ELSE SettlementType END AS ApplicationType
FROM 
(
    SELECT DISTINCT SE.ExposureKey, 
    SE.DueAmount,
    SE.CDI,
    SE.KeptPercentage AS KeptRate,
    SE.ImplementationDate, 
    CASE 
        WHEN SE.ImplementationDate IS NOT NULL AND dr.RestructuringDate IS NULL THEN SE.ImplementationDate 
        WHEN dr.RestructuringDate IS NOT NULL AND SE.ImplementationDate IS NULL THEN dr.RestructuringDate 
        WHEN SE.ImplementationDate >= dr.RestructuringDate THEN SE.ImplementationDate 
        WHEN dr.RestructuringDate > SE.ImplementationDate THEN dr.RestructuringDate 
    END AS ImplementationDateEver, 
    CASE 
        WHEN SE.ImplementationDate IS NOT NULL AND dr.RestructuringDate IS NULL THEN 'Settlement' 
        WHEN dr.RestructuringDate IS NOT NULL AND SE.ImplementationDate IS NULL THEN 'Restructuring' 
        WHEN SE.ImplementationDate >= dr.RestructuringDate THEN 'Settlement' 
        WHEN dr.RestructuringDate > SE.ImplementationDate THEN 'Restructuring' 
    END AS LatestImplementation,    
    SE.SettlementCDI, 
    SE.CountofImplementationsPerAccountBeforeLTSA,
    SE.CountofImplementationsPerAccountAfterLTSA,
    SE.HasActiveSettlementFlag, 
    SE.SettlementBucket,
    SE.ExposureStatus,
    SE.InitialAssignmentAmount,
    COALESCE(CAST(dr.RestructuringAmount AS FLOAT),0) as RestructuringAmount, 
    dr.RestructuringDiscountAmount,
    SE.DiscountAmount,
    SE.ProposedAmount, 
    COALESCE(CAST(SE.PrincipalProposedAmount AS FLOAT),0) as PrincipalProposedAmount,
    SE.RemainAmount,
    SE.SettlementAmount,
    SE.SettlementStatus,
    SE.NumOfInstalments,
    SE.IsActiveSettlement,
    SE.FirstInstalmentDate,
    SE.AppliedDate,
    SE.ApprovalDate,
    CASE
        WHEN SettlementType IN ('AB Migration', 'AB Migration (SOR_MGR)') THEN 'Migration Settlement'
        WHEN SettlementType IN ('CELL Ν3869 Διατραπεζική Επικοινωνία', 'L3869 AB Migration', 'L3869 AB Migration (SOR_MGR)', 'L3869 Court Decision', 'L3869 Court Decision (SOR_MGR)') THEN 'Migration L3869'
        WHEN SettlementType IN ('L3869 Temporary', 'L3869 Temporary (SOR_MGR)', 'L3869_Temporary', 'L3869_Temporary (SOR_MGR)') THEN 'Migration L3869 Temporary'
        WHEN SettlementType IN ('L3869_Approved', 'L3869_Approved (SOR_MGR)', 'Mercury_N.3869 Οριστική Απόφαση', 'Mercury_N.3869 Οριστική Απόφαση (SOR_MGR)', 'MRC Ν3869 Οριστική', 'MRC Ν3869 Οριστική (pending to Conf.Total amount)', 'MRC Ν3869 Οριστική (SOR_MGR)', 'Νόμος 3869 – Δικαστική Απόφαση', 'Νόμος 3869 - Δικαστική Απόφαση (SOR_MGR)', 'Νόμος 3869 – Bank – Περιορισμός Οφειλών', 'Νόμος 3869 – Διατραπεζική Επικοινωνία', 'Νόμος 3869 – 30.000 – Περιορισμός Οφειλών', 'Νόμος 3869 – Δικαστική απόφαση χωρίς ικανοπ. Alpha', 'Νόμος 3869 – Διατραπεζική Επικοινωνία', 'Νόμος 3869 – Δικαστική απόφαση ενεχομένων', 'Νόμος 3869 – Δικαστική απόφαση με ρευστοπ. Καλύμ.') THEN 'L3869 Approved'
        WHEN SettlementType IN ('L4605_Approved', 'L4605_Approved (SOR_MGR)') THEN 'Migration L4605 Approved'
        WHEN SettlementType IN ('L4605_Temporary', 'Νόμος 4605 - Προσωρινή Διαταγή') THEN 'Migration L4605 Temporary'
        WHEN SettlementType IN ('MRC Ν3869 Προσωρινή', 'MRC Ν3869 Προσωρινή (SOR_MGR)', 'MRC Ν3869 Προσωρινή (pending to Conf.Total amount)', 'Νόμος 3869 – Προσωρινή Διαταγή') THEN 'L3869 Pending'
        WHEN SettlementType IN ('MRC Ν3869 Expired', 'MRC Ν3869 Expired (SOR_MGR)') THEN 'L3869 with No Active Plan'
        WHEN SettlementType IN ('DPO within Limits', 'DPO within Limits (SOR_MGR)') THEN CASE WHEN SE.SubmissionDate > md.MigrationDate THEN 'Settlement' ELSE 'Migration' END
        WHEN SettlementType IN ('L.4738 OCW with HAIRCUT _ All Rels', 'L.4738 OCW with HAIRCUT _ All Rels (SOR_MGR)', 'Νόμος 4738 - Διακανονισμός Written Off Λογαριασμών') THEN 'OCW Approved'
        WHEN SettlementType IN ('Mercury Letter Offer', 'Mercury Letter Offer (SOR_MGR)') THEN 'Interim Period Settlement'
        WHEN SettlementType IN (
            'Mercury Out of Mandate', 'Mercury Out of Mandate (SOR_MGR)', 'Mercury Out of Mandate-AO Mass Approved', 'Mercury Out of Mandate-AO Mass Approved (SOR_MGR)', 'Mercury Preapproved as Declared', 'Mercury Preapproved as Declared (SOR_MGR)',
            'Mercury Preapproved with Documents', 'Mercury Preapproved with Documents (SOR_MGR)', 'Mercury Preapproved with Interview', 'Mercury Preapproved with Interview (SOR_MGR)', 'Mercury Reignition L3869', 'Mercury Reignition L3869 (SOR_MGR)', 
            'Out of Mandate', 'Out of Mandate (SOR_MGR)', 'Out_of_Mandate', 'Out_of_Mandate (SOR_MGR)', 'Preapproved_as_Declared', 'Preapproved_as_Declared (SOR_MGR)', 'Preapproved_with_Documents', 'Preapproved_with_Documents (SOR_MGR)' , 
            'Preapproved_with_Interview', 'Settlement Manual Correction', 'Settlement Manual Correction (SOR_MGR)',  'Venus Preapproved with Documents', 'Venus Preapproved as Declared',
            'Venus within Limits', 'Preapproved_with_Interview (SOR_MGR)', 'Διακανονισμός Οφειλής', 'Διακανονισμός Written Off Λογαριασμών', 'ΔΙΑΓΡΑΦΗ ΥΠΟΛΟΙΠΟΥ'
        ) THEN 'Cepal Settlement'
        WHEN SettlementType IN ('Άρση Εγγυητικής Σχέσης', 'Άρση Εγγυητικής Σχέσης (SOR_MGR)', 'ΑΡΣΗ ΕΓΓΥΗΣΗΣ') THEN 'Cepal Settlement on Guarantor'
    ELSE NULL
    END AS SettlementType, 
    CASE 
        WHEN SettlementType IN (
        'Περιορισμός Οφειλών με πληρωμένη Προκαταβολή',
        'Περιορισμός Οφειλών One Off',
        'Settlement Manual Correction (SOR_MGR)',
        'Citi Payment Arrangement',
        'AB Migration (SOR_MGR)',
        'Citi Payment Arrangement',
        'AB Migration',
        'Settlement Manual Correction'
        ) THEN NULL 
        WHEN 
        SettlementType ILIKE '%Νόμος%' OR 
        SettlementType ILIKE '%Νομος%' OR 
        SettlementType ILIKE '%L.%' OR 
        SettlementType ILIKE '%Ν.%' OR 
        SettlementType ILIKE '%N.%' OR 
        SettlementType ILIKE '%Law%' OR 
        SettlementType ILIKE '%3869%' OR 
        SettlementType ILIKE '%4224%' OR 
        SettlementType ILIKE '%4738%' OR 
        SettlementType ILIKE '%4605%' THEN 'Law'
        WHEN 
        SettlementType ILIKE '%Out_of%' OR 
        SettlementType ILIKE '%Out of%' OR 
        SettlementType IN ('ΑΡΣΗ ΕΓΓΥΗΣΗΣ', 'Άρση Εγγυητικής Σχέσης', 'Άρση Εγγυητικής Σχέσης (SOR_MGR)') 
        THEN 'Out of mandate'
        WHEN SettlementType IS NOT NULL THEN 'In mandate'
    END AS SettlementTypeGroup,
    CASE WHEN  SE.IsMigratedFlag IS TRUE THEN 1 ELSE 0 END AS SettlementFromMigration,
    I.NoOfInstallments as NumberOfInstallments,
    I.PastInstallments AS NumberOfPastInstallments,
    I.OverdueInstallments AS NumberOfMissedInstallments, 
    I.FirstInstallmentDate,
    I.MonthlyInstallmentAmount,
    I.DownPaymentFlagForApplied,
    I.FirstInstallmentDateAfterApplication,
    I.ScheduledAmount
    FROM settlementexposure SE
    LEFT JOIN MigrationDate md 
    on SE.PortfolioName = md.PortfolioName 
    LEFT JOIN restructuring dr 
    ON SE.Exposurekey = dr.NewExposureKey 
    LEFT JOIN installments_V0 I
    ON SE.ExposureKey = I.ExposureKey
) SE 
LEFT JOIN dunderlaw_active B
ON SE.CDI = B.CDI
               
""")
settlement_fields.createOrReplaceTempView("settlement_fields")

In [0]:
%sql
select * from settlement_fields where exposurekey like '%989008122798300'

## Collections

In [0]:
Collections = spark.sql(f"""
SELECT c.PortfolioName, 
    CASE WHEN LEFT(c.ExposureKey,3) = P.Prefix THEN c.ExposureKey ELSE CONCAT(P.Prefix,c.ExposureKey) END ExposureKey, 
    MAX(ActualDate) AS LastPaymentDate, 
    SUM(CASE WHEN COALESCE(MatchedDate, '{run_date}') BETWEEN '{FirstWorkingDate}' AND '{SecondWorkingDateNextMonth}' AND ActualDate BETWEEN '{FirstWorkingDate}' AND '{run_date}' AND NOT (ActualDate < '{FirstWorkingDate}' and MatchedDate IN ('{FirstWorkingDate}', '{SecondWorkingDateCurrentMonth}')) then Amount else NULL END) as MonthlyPayments,
    SUM(CASE WHEN COALESCE(MatchedDate, '{run_date}') BETWEEN md.CutOffDate AND '{SecondWorkingDateNextMonth}' THEN Amount ELSE NULL END) AS TotalPaymentsAfterLTSA,
    SUM(CASE WHEN COALESCE(MatchedDate, '{run_date}') < md.CutOffDate THEN Amount ELSE NULL END) AS TotalPaymentsBeforeLTSA
FROM 
    (   
        SELECT DISTINCT   c.PortfolioName, md.MigrationDate, CollectionSourceKey, ExposureKey, ActualDate, Amount, MatchedDate, RANK() OVER (PARTITION BY Exposurekey ORDER BY CASE WHEN MatchedDate IS NOT NULL THEN 1 ELSE 2 END ASC)  AS rank  
        FROM `{env}-cdh`.derivative.dcollection c
        LEFT JOIN MigrationDate as md on md.portfolioname = c.portfolioname
        WHERE (
        ActualDate <= '{run_date}' 
        AND BusinessDate BETWEEN '{run_date}' AND '{SecondWorkingDateNextMonth}'
        AND CAST(CASE WHEN MatchedDate IS NULL AND FinalReason = 'Matched' THEN '{run_date}' ELSE MatchedDate END AS DATE) <= '{SecondWorkingDateNextMonth}' 
        ) 
    ) as c
LEFT JOIN `{env}-cdh`.primary.portfolio as p on c.PortfolioName = p.PortfolioName AND p.BusinessDate='{run_date}'
LEFT JOIN MigrationDate as md on md.portfolioname = c.portfolioname
WHERE Rank = 1
GROUP BY c.PortfolioName, CASE WHEN LEFT(c.ExposureKey,3) = P.Prefix THEN c.ExposureKey ELSE CONCAT(P.Prefix,c.ExposureKey) END
""")
Collections.createOrReplaceTempView("Collections")

## Assignment

In [0]:
assignment = spark.sql(f"""
SELECT DISTINCT PortfolioName, ExposureKey, 
    CDI, 
    case when AssignmentProcess = 'CA/FA/Sme Officer' THEN Channel 
        WHEN LOWER(AssignmentProcess) LIKE '%paid off%' THEN 'Paid-off'
        WHEN AssignmentProcess = 'CA Dialer' AND AgentFullName IS NULL THEN 'CA Dialer'
        when AssignmentProcess in ('Collection', 'Collection_Judicial_Escalation')  and ExternalCompanyFullName is not null then concat(ExternalCompanyFullName, ' Collection')
        when AssignmentProcess in ('Lf_Offer') and ExternalCompanyFullName is not null then concat(ExternalCompanyFullName, ' Offer')
        else 'Unhandled' 
    end as AssignedChannel, ProcessType
    from `{env}-cdh`.primary.assignment
    WHERE PortfolioName IN {PortfolioName} AND BusinessDate = '{run_date}'
    and AssignmentType = 'Channel Assignment'
""")

assignment.createOrReplaceTempView("assignment")

## L3869 fields

In [0]:
dunderlaw_maxbusinessdate = spark.sql(f"""
SELECT * FROM (
        SELECT a.*, row_number() OVER (PARTITION BY a.CDI ORDER BY a.BusinessDate DESC) AS rn /***** TO BE REMOVED WHEN L3869 issues are resolved! *****/
        FROM `{env}-cdh`.derivative.dunderlaw A
        INNER JOIN (    
                SELECT CDI, MAX(BusinessDate) AS BusinessDate
                FROM `{env}-cdh`.derivative.dunderlaw A
                WHERE BusinessDate <= '{run_date}'
                AND PortfolioName IN {PortfolioName}
                AND UnderLawType = 'N3869'  
                and sourcesystem = 'process'   
                group by CDI  
                ) AS B ON A.CDI = B.CDI AND A.BusinessDate = B.BusinessDate
        WHERE PortfolioName IN {PortfolioName}
        AND UnderLawType = 'N3869'    
        and sourcesystem = 'process'       
) WHERE rn = 1 
""")

dunderlaw_maxbusinessdate.createOrReplaceTempView("dunderlaw_maxbusinessdate")

# In case an application has UnderlawType = 'N3869' without receiving label InlawFlag = 0, then we consider it as current L3869.
isPrevEoML3869 = spark.sql(
f'''
SELECT *
FROM (
    SELECT 
            CDI,
            InLawFlag,
           UnderLawType,
            ROW_NUMBER() OVER (
                    PARTITION BY CDI
                    ORDER BY BusinessDate DESC
            ) AS rn
    FROM `{env}-cdh`.derivative.dunderlaw
    WHERE BusinessDate <= '{PreviousMonthLastWorkingDate}'
    AND PortfolioName IN {PortfolioName}
    AND UnderLawType LIKE '%3869%'
    and sourcesystem = 'process'    
) t
WHERE rn = 1
''')
isPrevEoML3869.createOrReplaceTempView("isPrevEoML3869")

isCurrentL3869 = spark.sql(
f'''
SELECT *
FROM (
    SELECT 
            CDI,
            InLawFlag,
           UnderLawType,
            ROW_NUMBER() OVER (
                    PARTITION BY CDI
                    ORDER BY BusinessDate DESC
            ) AS rn
    FROM `{env}-cdh`.derivative.dunderlaw
    WHERE BusinessDate <= '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND UnderLawType LIKE '%3869%'
    and sourcesystem = 'process'      
) t
WHERE rn = 1
''')
isCurrentL3869.createOrReplaceTempView("isCurrentL3869")

applicationstatusdate = spark.sql(f"""
                    SELECT DISTINCT CDI, ApplicationStatusDate
                    FROM dunderlaw_maxbusinessdate A
                    WHERE LawStatus in ('Εκτός Ν3869', 'Εκτός Ν3869 - Απορριπτική Δικαστ. Απόφαση', 'Εκτός Ν3869 - Παραίτηση', 'Εκτός ν3869 - Ματαίωση')
""")
applicationstatusdate.createOrReplaceTempView("applicationstatusdate")

prvapplicationstatusdate = spark.sql(f"""
                    SELECT A.CDI, MAX(BusinessDate) AS LatestBusinessDate
                    FROM `{env}-cdh`.derivative.dparty A
                    INNER JOIN applicationstatusdate B
                    ON A.CDI = B.CDI
                    WHERE A.PortfolioName IN {PortfolioName}
                    AND BusinessDate < ApplicationStatusDate
                    GROUP BY A.CDI
""")
prvapplicationstatusdate.createOrReplaceTempView("prvapplicationstatusdate")

prvsegmentation = spark.sql(f"""
                    SELECT DISTINCT A.CDI, Segment AS PreviousSegmentation
                    FROM `{env}-cdh`.derivative.dparty A
                    INNER JOIN prvapplicationstatusdate B
                    ON A.CDI = B.CDI
                    AND A.BusinessDate = B.LatestBusinessDate
                    WHERE A.PortfolioName IN {PortfolioName}
""")

prvsegmentation.createOrReplaceTempView("prvsegmentation")

dunderlaw_history = spark.sql(f"""
select CDI, MAX(CASE WHEN BusinessDate < '{run_date}' AND UnderLawType = 'N3869' AND InlawFlag = 1 THEN 1 ELSE 0 END) AS L3869PrevFlag
from `{env}-cdh`.derivative.dunderlaw
WHERE 1=1
AND PortfolioName IN {PortfolioName}
and sourcesystem = 'process'     
GROUP BY CDI
""")

dunderlaw_history.createOrReplaceTempView("dunderlaw_history")

dunderlaw = spark.sql(f"""
SELECT DISTINCT A.CDI, 
        CASE WHEN UnderLawType = 'N3869' THEN LawStatus ELSE NULL END AS L3869Status1,
        CASE WHEN UnderLawType = 'N3869' AND LawStatus IN ('Εκτός Ν3869', 'Εκτός Ν3869 - Απορριπτική Δικαστ. Απόφαση',
        'Εκτός Ν3869 - Παραίτηση', 'Εκτός ν3869 - Ματαίωση') THEN date_format(ApplicationStatusDate, 'MMMM-yyyy') ELSE NULL END AS L3869MonthExit,
        CASE WHEN UnderLawType = 'N3869' THEN CourtOutcome ELSE NULL END AS L3869CourtDecision,
        ApplicationStatus,
        CASE WHEN UnderLawType = 'N3869' AND LawStatus IS NOT NULL AND CourtOutcome IS NOT NULL THEN CourtDate ELSE NULL END AS L3869FinalCourtDate,
        D.PreviousSegmentation,
        SubmissionDate as L3869SubmissionDate,
        CASE WHEN UnderLawType = 'N3869' AND LawStatus = 'Εκτός Ν3869 - Παραίτηση' THEN ApplicationStatusDate ELSE NULL END AS L3869ResignationDate,
        CASE WHEN ApplicationStatus = 'Expelled' AND UnderLawType = 'N3869' THEN 'Yes' ELSE 'No' END AS EkptosiFlag,             -- Currently the value 'Expelled' does not exist for Hoist Portfolios
        CASE WHEN ApplicationStatus = 'Expelled' AND UnderLawType = 'N3869' THEN LawStatus ELSE NULL END AS EkptosiStatus 
FROM dunderlaw_maxbusinessdate A
LEFT JOIN prvsegmentation D
ON A.CDI = D.CDI
""")
dunderlaw.createOrReplaceTempView("dunderlaw")

## Letter Flag

In [0]:
letterflagexist = spark.sql(f"""
                    SELECT DISTINCT A.CDI AS LetterFlagCDI
                    FROM dexposure A
                    INNER JOIN LetterFlag B
                    ON A.PartyTaxRegistrationNumber = B.AFM
""")
letterflagexist.createOrReplaceTempView("letterflagexist")

## Contact fields

##### Activities

In [0]:
interaction_v0 = spark.sql(
f'''
SELECT DISTINCT   
    CASE WHEN LEFT(intr.ExposureKey,3) = md.Prefix THEN intr.ExposureKey ELSE CONCAT(md.Prefix,intr.ExposureKey) END ExposureKey,
    intr.PortfolioName,
    CASE WHEN LEFT(intr.CDI,3) = md.Prefix THEN intr.CDI ELSE CONCAT(md.Prefix,intr.CDI) END CDI,
    INTR.InteractionTypeCode,
    INTR.INTERACTIONTYPE,
    CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
    INTR.InteractionKey,
    INTR.INTERACTIONTYPEKEY,
    intr.InteractionCategory,
    intr.FirstLevelCategory,
    intr.SecondLevelCategory,
    intr.AttemptOutcome,
    intr.IsPositiveFlag,
    intr.IsPromiseFlag,
    intr.ContactType,
    intr.IsCallFlag,
    intr.AttemptType,
    intr.IsFtFAppointmentFlag,
    intr.createddate,
    intr.PortfolioName, 
    intr.SourceSystem, 
    md.CutOffDate
from 
(
SELECT * FROM `{env}-cdh`.primary.get_interactions_by_business_date('{DecomissionDate}') intr
WHERE InteractionDate <= '{DecomissionDate}'
and intr.PortfolioName IN {PortfolioName}

UNION ALL 

SELECT * FROM `{env}-cdh`.primary.get_interactions_by_business_date('{run_date}') intr
WHERE InteractionDate > '{DecomissionDate}' AND InteractionDate <= '{run_date}'
and intr.PortfolioName IN {PortfolioName}
) intr 
left join MigrationDate md 
on md.PortfolioName = intr.PortfolioName
where 1=1
and intr.PortfolioName IN {PortfolioName}

''')
interaction_v0.createOrReplaceTempView("interaction_v0")

##### LTSA

In [0]:
 LTSA = spark.sql(f"""
    SELECT *, ROW_NUMBER() over(partition by LoanId order by cast(LastAttemptDatebeforeLTSA as date) desc) rnk
    FROM 
    (
    select  BorrowerCode, 
    LoanCode, 
    LoanId, 
    RPCbeforeLTSA, 
    RPCGuarantorbeforeLTSA, 
    AttemptbeforeLTSA,
    AttemptGuarantorbeforeLTSA,
    LastRPCStatusbeforeLTSA,
    CASE WHEN LastRPCDatebeforeLTSA = 'NULL' THEN NULL ELSE cast(LastRPCDatebeforeLTSA as date) END as LastRPCDatebeforeLTSA,
    CASE WHEN LastAttemptDatebeforeLTSA = 'NULL' THEN NULL ELSE cast(LastAttemptDatebeforeLTSA as date) END as LastAttemptDatebeforeLTSA
    from `{env}-reportinputfiles`.investor.hoist_BeforeLTSA
    )
""")
LTSA.createOrReplaceTempView("LTSA")

##### eap

In [0]:
eap = spark.sql(f"""
WITH raw_eap AS (

  -- interactions (qc) enriched with AgreementKey
  SELECT intr.PortfolioName,
      regexp_replace(intr.ExposureKey, '[^0-9]', '') as ExposureKey,
      exp.AgreementKey,                     
      intr.InteractionDate      AS Last_EAP_Date,
      intr.InteractionTypeCode  AS ActivityTypeKey,
      CAST(NULL AS STRING)      AS CompletionResult,
      intr.SourceSystem         AS SourceSystem
  FROM interaction_v0 intr
  LEFT JOIN `{env}-cdh`.primary.exposure exp
  ON regexp_replace(intr.ExposureKey, '[^0-9]', '') = regexp_replace(exp.ExposureKey, '[^0-9]', '')
  AND intr.PortfolioName = exp.PortfolioName
  AND exp.BusinessDate = intr.InteractionDate
  WHERE (intr.InteractionTypeCode IN ('5196','5199','5200','5201','6542') AND intr.SourceSystem = 'qc')
    OR (intr.InteractionTypeKey IN ( '8781') AND intr.SourceSystem = 'qca') -- InteractionTypeKey = 8781 but for Pre decommission

  UNION ALL

  -- legalaction (qc, by ExposureKey)
  SELECT la.PortfolioName,
      regexp_replace(la.ExposureKey, '[^0-9]', '') as ExposureKey,
      la.AgreementKey,
      la.LegalActionDate        AS Last_EAP_Date,
      la.LegalActionTypeCode    AS ActivityTypeKey,
      CAST(NULL AS STRING)      AS CompletionResult,
      la.SourceSystem           AS SourceSystem
  FROM `{env}-cdh`.primary.legalaction la
  WHERE (la.LegalActionTypeCode IN ('5196','5199','5200','5201','6542')
    AND la.LegalActionDate <= '{run_date}'
    AND la.SourceSystem = 'qc')
    or (la.LegalActionTypeCode IN  ( '8781') 
    AND la.LegalActionDate <= '{run_date}'
    AND la.SourceSystem = 'qca')  

  UNION ALL

  -- legalaction (arotron, via CDI)
  SELECT la.PortfolioName,
      exp.ExposureKey,
      exp.AgreementKey,
      la.LegalActionDate        AS Last_EAP_Date,
      la.LegalActionTypeCode    AS ActivityTypeKey,
      la.CompletionResult       AS CompletionResult,
      la.SourceSystem           AS SourceSystem
  FROM `{env}-cdh`.primary.legalaction la
  INNER JOIN `{env}-cdh`.primary.exposure exp
    ON regexp_replace(la.CDI, '[^0-9]', '') = regexp_replace(exp.CDI, '[^0-9]', '')
   AND exp.BusinessDate = la.BusinessDate
   AND exp.PortfolioName = la.PortfolioName
  WHERE  UPPER(la.LegalType) like '%ΕΑΠ%'
    AND la.LegalActionDate <= '{run_date}'
    AND la.BusinessDate = '{run_date}'
    AND la.SourceSystem = 'arotron'
),

EAP_CTE AS (
  SELECT PortfolioName,
      ExposureKey,
      AgreementKey,
      Last_EAP_Date,
      ActivityTypeKey,
      CompletionResult,
      SourceSystem,
      ROW_NUMBER() OVER (
          PARTITION BY AgreementKey
          ORDER BY Last_EAP_Date DESC, PortfolioName DESC, ExposureKey DESC
      ) AS RowNumber
  FROM raw_eap
)
SELECT DISTINCT
    EAP_CTE.AgreementKey,
    EAP_CTE.Last_EAP_Date,
    CASE 
      WHEN (EAP_CTE.SourceSystem = 'qc'
            AND EAP_CTE.ActivityTypeKey IN ('5199','5200')) THEN 'Completed'
      WHEN (EAP_CTE.SourceSystem = 'arotron'
            AND EAP_CTE.CompletionResult IS NOT NULL) THEN 'Completed'
      ELSE 'In Progress'
    END AS EAP_Status,
  CASE WHEN EAP_CTE.Last_EAP_Date > md.MigrationDate THEN 'Yes' ELSE 'No' END as EAPDateAfterMigration
FROM EAP_CTE 
LEFT JOIN MigrationDate md on eap_cte.PortfolioName = md.PortfolioName
WHERE EAP_CTE.RowNumber = 1
""")
eap.createOrReplaceTempView("eap")


In [0]:
re = spark.sql(f"""  
    SELECT DISTINCT
        dexp.AgreementKey,
        CASE WHEN  floor(months_between('{run_date}', re.InteractionDate) / 12) < 3 THEN 'Yes' ELSE 'N/A' END AS Positive_RE_Exists,
        RANK() OVER (PARTITION BY dexp.AgreementKey ORDER BY CASE WHEN floor(months_between('{run_date}', re.InteractionDate) / 12) < 3 THEN 1 ELSE 0 end desc)  AS rank
    FROM `{env}-cdh`.primary.exposure  dexp 
    INNER JOIN interaction_v0 re
    ON regexp_replace(dexp.ExposureKey, '[^0-9]', '') = regexp_replace(re.ExposureKey, '[^0-9]', '')
    AND re.InteractionTypeCode = '5200'
    AND re.SourceSystem = 'qc'    
    AND re.PortfolioName = dexp.PortfolioName
    WHERE dexp.PortfolioName IN {PortfolioName} 
""")
re.createOrReplaceTempView("re")

In [0]:
interaction = spark.sql(f"""
   SELECT intr.*,  CASE WHEN dprim.CDI IS NOT NULL THEN 1 ELSE 0 END AS PrimaryActivity,
   CASE   
      WHEN CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and (AttemptOutcome = 'negative' OR InteractionType LIKE '%refuse%' OR SecondLevelCategory LIKE '%refuse%') THEN 'Negative'  
      WHEN CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND intr.FirstLevelCategory = '4. Negative Communication' THEN 'Negative' 
      WHEN CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and  (AttemptOutcome = 'positive' OR InteractionType LIKE '%promise%' /*OR isPositiveFlag = TRUE */ OR isPromiseFlag = TRUE)   THEN 'Positive'
      WHEN  CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND intr.FirstLevelCategory = '5.Positive Communication' THEN 'Positive'
      END AS InteractionTypeFlag
   FROM (
        SELECT DISTINCT
           intr.ExposureKey,
           intr.CDI,
           INTR.InteractionTypeCode,
           INTR.INTERACTIONTYPE,
           CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
           INTR.InteractionKey,
           INTR.INTERACTIONTYPEKEY,
           intr.InteractionCategory,
           intr.SecondLevelCategory,
           intr.AttemptOutcome,
           intr.IsPositiveFlag,
           intr.IsPromiseFlag,
           intr.ContactType,
           intr.IsCallFlag,
           intr.AttemptType,
           intr.IsFtFAppointmentFlag,
           intr.createddate,
           intr.PortfolioName, 
           intr.SourceSystem,
           intr.FirstLevelCategory
        FROM interaction_v0 intr   

        UNION 

        SELECT DISTINCT
           de.ExposureKey,
           intr.CDI,
           INTR.InteractionTypeCode,
           INTR.INTERACTIONTYPE,
           CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
           INTR.InteractionKey,
           INTR.INTERACTIONTYPEKEY,
           intr.InteractionCategory,
           intr.SecondLevelCategory,
           intr.AttemptOutcome,
           intr.IsPositiveFlag,
           intr.IsPromiseFlag,
           intr.ContactType,
           intr.IsCallFlag,
           intr.AttemptType,
           intr.IsFtFAppointmentFlag,
           intr.createddate,
           intr.PortfolioName,
           intr.SourceSystem,
           intr.FirstLevelCategory
        FROM interaction_v0 intr    
        INNER JOIN (select distinct exposurekey, CDI from dexposure) de
        ON de.CDI = intr.CDI
   ) intr
   LEFT JOIN (select exposurekey, CDI from dexposure) dprim  ON intr.ExposureKey = dprim.ExposureKey AND INTR.CDI = dprim.CDI 
""")
interaction.createOrReplaceTempView("interaction")

In [0]:
Activities = spark.sql(f"""
   SELECT DISTINCT
      intr.ExposureKey,
      INTR.InteractionTypeCode,
      INTR.INTERACTIONTYPE,
      CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
      INTR.InteractionKey,
      INTR.INTERACTIONTYPEKEY,
      intr.AttemptOutcome,
      intr.ContactType,
      MD.cutoffdate,
      CASE WHEN InteractionTypeFlag = 'Negative' AND CAST(INTR.InteractionDate AS DATE) >= MD.cutoffdate THEN 1 ELSE 0 END AS NegativeAfterCutOff,
      CASE when CAST(INTR.InteractionDate AS DATE) < MD.cutoffdate THEN 0
            WHEN CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and 
            (
               UPPER(COALESCE(ContactType, '')) IN ('RIGHT PARTY CONTACT', 'FACE TO FACE', 'RIGHT PERSON CONTACT') OR LOWER(COALESCE(InteractionType, '')) LIKE '%planning business trip%'
            ) 
            THEN 1 
            WHEN CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory NOT IN ('1. no contact', '2.contact - Not RPC'))
            THEN 1
            ELSE 0 END AS RPC, 
      CASE when CAST(INTR.InteractionDate AS DATE) < MD.cutoffdate THEN 0
            WHEN ER.ExposureKey IS  NULL THEN 0
            WHEN CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and 
            (
               UPPER(COALESCE(ContactType, '')) IN ('RIGHT PARTY CONTACT', 'FACE TO FACE', 'RIGHT PERSON CONTACT') OR LOWER(COALESCE(InteractionType, '')) LIKE '%planning business trip%'
            ) 
            THEN 1 
            WHEN CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory NOT IN ('1. no contact', '2. contact - Not RPC'))
            THEN 1
         ELSE 0 END AS GuarantorRPC,  
      CASE WHEN InteractionTypeFlag = 'Positive' and CAST(INTR.InteractionDate AS DATE) >= MD.cutoffdate  THEN 1 ELSE 0 END AS Positive, 
      CASE WHEN  InteractionTypeFlag = 'Negative' and CAST(INTR.InteractionDate AS DATE) >= MD.cutoffdate  THEN 1 ELSE 0 END AS Negative, 
      CASE WHEN CAST(INTR.InteractionDate AS DATE) < MD.cutoffdate THEN 0
           WHEN  intr.InteractionTypeCode IN ('1020','6028','3070','3086','3094') and CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' THEN 1 
           WHEN  intr.InteractionTypeCode IN ('8693','8614','8743','7020','8751') and CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' THEN 1 
         ELSE 0 END AS Inbound,
      PrimaryActivity
   FROM interaction intr 
   LEFT JOIN MigrationDate as md on md.portfolioname = intr.portfolioname
   LEFT JOIN (SELECT DISTINCT ExposureKey, CDI, PortfolioName
                    FROM ExposureRelations 
                    WHERE PartyRelationshipType <> 'Primary Owner'
   ) AS ER 
   ON regexp_replace(intr.ExposureKey, '[^0-9]', '') = ER.ExposureKey AND regexp_replace(intr.CDI, '[^0-9]', '') = ER.CDI and intr.PortfolioName = er.PortfolioName
""")
Activities.createOrReplaceTempView("Activities")

In [0]:
ActivitiesRanked = spark.sql(f""" 
    SELECT DISTINCT
        intr.ExposureKey,
        INTR.InteractionTypeCode,
        INTR.INTERACTIONTYPE,
        CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
        INTR.InteractionKey,
        intr.AttemptOutcome,
        MD.cutoffdate,
        CASE WHEN InteractionTypeFlag = 'Negative' AND CAST(INTR.InteractionDate AS DATE) >= MD.cutoffdate and PrimaryActivity = 1 THEN 
            RANK() OVER (PARTITION BY INTR.ExposureKey ORDER BY CASE WHEN InteractionTypeFlag = 'Negative' AND CAST(INTR.InteractionDate AS DATE) >= MD.cutoffdate THEN 1 else 0 end desc, INTR.InteractionDate DESC, INTR.CREATEDDATE DESC, INTR.InteractionKey desc)   
            ELSE -1 
        END AS NegativeAfterCutOffRank,
        CASE when CAST(INTR.InteractionDate AS DATE) < MD.cutoffdate THEN -1
            WHEN (CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and 
                (UPPER(COALESCE(ContactType, '')) IN ('RIGHT PARTY CONTACT', 'FACE TO FACE', 'RIGHT PERSON CONTACT') OR LOWER(COALESCE(InteractionType, '')) LIKE '%planning     business trip%')) 
                or 
                (CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory NOT IN ('1. no contact', '2. contact - Not RPC')))
                THEN RANK() OVER (PARTITION BY INTR.ExposureKey ORDER BY CASE WHEN (CAST(intr.InteractionDate AS DATE) <= '{DecomissionDate}' and 
                    (
                        UPPER(COALESCE(ContactType, '')) IN ('RIGHT PARTY CONTACT', 'FACE TO FACE', 'RIGHT PERSON CONTACT') OR LOWER(COALESCE(InteractionType, '')) LIKE '%planning business trip%'
                        )) 
                            or 
                    (CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory NOT IN ('1. no contact', '2. contact - Not RPC')))
                    THEN  1 else 0 end desc, INTR.InteractionDate DESC, INTR.CREATEDDATE DESC, INTR.InteractionKey desc, case when intr.attemptoutcome is not null then 1 else 0 end desc)   
                ELSE -1 END AS RPCRank 
    FROM interaction intr
    LEFT JOIN MigrationDate as md on md.portfolioname = intr.portfolioname
""")
ActivitiesRanked.createOrReplaceTempView("ActivitiesRanked")

In [0]:
ActivitiesGroupped = spark.sql(f"""
SELECT
    intr.ExposureKey,
    MIN(CASE WHEN NegativeAfterCutOff = 1 and PrimaryActivity = 1 THEN InteractionDate ELSE NULL END ) AS DateofFirstRejectedOffer, 
    MAX(CASE WHEN NegativeAfterCutOff = 1 and PrimaryActivity = 1 THEN InteractionDate ELSE NULL END ) AS DateofLastRejectedOffer,
    COUNT(DISTINCT CASE WHEN NegativeAfterCutOff = 1 and PrimaryActivity = 1 THEN CONCAT(InteractionTypeCode, InteractionDate) ELSE NULL END ) AS TotalCountOfOffersRejected,
    MIN(CASE WHEN RPC = 1 and PrimaryActivity = 1 THEN InteractionDate ELSE NULL END ) AS DateOfFirstRPC,  
    MAX(CASE WHEN RPC = 1 and PrimaryActivity = 1 THEN InteractionDate ELSE NULL END ) AS DateOfLastRPC, 
    COUNT(DISTINCT CASE WHEN RPC = 1 and PrimaryActivity = 1 THEN CONCAT(InteractionTypeCode, InteractionDate) ELSE NULL END ) AS TotalCountOfRPCs,
    COUNT(DISTINCT CASE WHEN Positive = 1 and PrimaryActivity = 1 THEN CONCAT(InteractionTypeCode, InteractionDate) ELSE NULL END ) AS PositiveRPC,
    COUNT(DISTINCT CASE WHEN Negative = 1 and PrimaryActivity = 1 THEN CONCAT(InteractionTypeCode, InteractionDate) ELSE NULL END ) AS NegativeRPC,
    max(CASE WHEN Inbound = 1 and PrimaryActivity = 1 THEN InteractionDate ELSE NULL END ) AS LastInboundDate, 
    MAX(CASE WHEN GuarantorRPC = 1 THEN InteractionDate ELSE NULL END ) AS DateOfLastGuarantorRPC,
    COUNT(DISTINCT CASE WHEN GuarantorRPC = 1 THEN CONCAT(InteractionTypeCode, InteractionDate) ELSE NULL END ) AS TotalCountOfGuarantorRPC       
FROM Activities intr 
GROUP BY ExposureKey
""")
ActivitiesGroupped.createOrReplaceTempView("ActivitiesGroupped")

##### Activities Dialer

In [0]:
Dialer_v0 = spark.sql(
f'''
select DISTINCT  
      de.ExposureKey
    , cast('' as string) as InteractionTypeKey 
    , cast(D.DateHour as date ) as InteractionDate
    , D.DateHour as CreatedDate    
    , d.DialerKey AS InteractionKey
    , de.CDI
    , d.CallDisposition
    , d.PortfolioName
from `{env}-cdh`.primary.dialer d
join ( SELECT DISTINCT ExposureKey, casecode, CDI 
       FROM `{env}-cdh`.primary.exposure
       WHERE sourcesystem IN ( 'qc', 'qca')         
) AS de 
on cast(d.casecode as string) = cast(de.casecode as string) 
WHERE D.PortfolioName IN {PortfolioName}
AND D.BusinessDate = (SELECT MAX(BusinessDate) FROM `{env}-cdh`.primary.dialer WHERE PortfolioName IN {PortfolioName})
AND d.CallDisposition IS NOT NULL
AND d.CallDisposition <> ''
AND cast(D.DateHour as date ) <= '{run_date}'
'''
)
Dialer_v0.createOrReplaceTempView("Dialer_v0")

In [0]:
ActivityAttemptsDialer = spark.sql(f'''

SELECT  A.* 
      , CASE WHEN ER.ExposureKey IS NOT NULL THEN 1 ELSE 0 END AS GuarantorAttemptEver 
      , CASE WHEN CAST(a.InteractionDate AS DATE) < MD.cutoffdate THEN 1 ELSE 0 END AS AttemptBeforeCutOff
      , CASE -- Pre Decomission 
          WHEN CAST(A.InteractionDate AS DATE) Between MD.cutoffdate and '{DecomissionDate}'
          and (A.ContactType IN ('Right Person Contact', '3rd Person Contact', 'RIGHT PARTY CONTACT','Dialer Answer') OR LOWER(COALESCE(A.InteractionType, '')) LIKE '%planning business trip%' )
          THEN 1  -- Post Decomission
          WHEN CAST(A.InteractionDate AS DATE) > '{DecomissionDate}' AND (A.FirstLevelCategory <> '1. no contact' or A.ContactType = 'Dialer Answer')
          THEN 1   
          ELSE 0 
          END AS ContactAfterLTSA, 
       CASE WHEN dprim.CDI IS NOT NULL THEN 1 ELSE 0 END AS PrimaryActivity 
FROM (                                   
        SELECT 
            intr.PortfolioName,
            intr.ExposureKey,
            intr.CDI,
            CAST( intr.InteractionDate AS DATE) InteractionDate,
            intr.InteractionKey,
            intr.InteractionType,
            INTR.InteractionTypeCode,
            intr.ContactType,
            intr.InteractionCategory,
            INTR.FirstLevelCategory
        FROM interaction intr  
        WHERE 1=1    
            AND    UPPER(COALESCE(ContactType, '')) <> 'SMS'
            AND (
                intr.ContactType IN ('Right Person Contact', '3rd Person Contact','RIGHT PARTY CONTACT') 
                OR COALESCE(intr.IsCallFlag, false) = true
                OR COALESCE(intr.IsFtFAppointmentFlag, false) = true
                OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%appointment%'
                OR LOWER(COALESCE(intr.AttemptType, '')) LIKE '%unsuccess%'
                OR LOWER(COALESCE(intr.AttemptOutcome, '')) LIKE '%no answer%'
                OR LOWER(COALESCE(intr.AttemptOutcome, '')) LIKE '%wrong%'
                OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%planning business trip%'
                OR intr.interactionType = 'Dialer Attempts (SIP)' 
                OR 
                (FirstLevelCategory <> '0. Legal Action - no contact' 
                AND NOT (FirstLevelCategory = "1. no contact" AND SecondLevelCategory = "1.1 no contact - no attempt")
                )
                )
            
        UNION 
        SELECT
            di.PortfolioName,
            di.ExposureKey,
            di.CDI, 
            di.InteractionDate,
            di.InteractionKey,
            'Dialer' as InteractionType,
            'Dialer' as InteractionTypeCode,
            CASE WHEN di.CallDisposition IN ('ANSWERED', 'ANSWER') THEN 'Dialer Answer' else null end as ContactType,
            'Dialer' AS InteractionCategory,
            '' AS FirstLevelCategory
        FROM Dialer_v0 as di
) AS A 
LEFT JOIN MigrationDate as md on md.portfolioname = a.portfolioname
LEFT JOIN (SELECT DISTINCT ExposureKey, CDI, PortfolioName
                    FROM ExposureRelations 
                    WHERE PartyRelationshipType <> 'Primary Owner') AS ER        ON regexp_replace(a.ExposureKey, '[^0-9]', '') = ER.ExposureKey AND regexp_replace(a.CDI, '[^0-9]', '') = ER.CDI and a.PortfolioName = er.PortfolioName
LEFT JOIN (select exposurekey, CDI from dexposure) dprim  ON a.ExposureKey = dprim.ExposureKey AND a.CDI = dprim.CDI 

''')
ActivityAttemptsDialer.createOrReplaceTempView("ActivityAttemptsDialer")

In [0]:
ActivityAttemptsDialerGroupped = spark.sql(f"""
SELECT
    intr.ExposureKey,          
    MAX(CASE WHEN GuarantorAttemptEver = 1 AND AttemptBeforeCutOff = 0  THEN intr.InteractionDate ELSE NULL END ) AS DateOfLastGuarantorAttempt,
    COUNT(DISTINCT CASE WHEN GuarantorAttemptEver = 1 AND AttemptBeforeCutOff = 0  THEN CONCAT(intr.InteractionTypeCode, intr.InteractionDate) ELSE NULL END ) AS TotalCountOfGuarantorAttempts,
    COUNT(DISTINCT CASE WHEN GuarantorAttemptEver = 1  AND AttemptBeforeCutOff = 0  THEN intr.CDI ELSE NULL END ) AS DistinctCountAttemptedGuarantors,
    Min(CASE WHEN AttemptBeforeCutOff = 0 and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DateofFirstAttemptAfterLTSA,
    MAX(CASE WHEN AttemptBeforeCutOff = 0 and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DateOfLastAttemptAfterLTSA,          
    COUNT(DISTINCT CASE WHEN AttemptBeforeCutOff = 0 and PrimaryActivity = 1 THEN CONCAT(intr.InteractionTypeCode, intr.InteractionDate) ELSE NULL END ) AS TotalCountOfAttempts_after_LTSA,
    MAX(CASE WHEN AttemptBeforeCutOff = 1 AND intr.PortfolioName = 'Mercury' and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DateOfLastAttemptBeforeLTSA,
    COUNT(DISTINCT CASE WHEN AttemptBeforeCutOff = 0 and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DistinctAttemptDays_after_LTSA,  
    Min(CASE WHEN ContactAfterLTSA = 1  and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DateOfFirstContact, 
    MAX(CASE WHEN ContactAfterLTSA = 1  and PrimaryActivity = 1 THEN intr.InteractionDate ELSE NULL END ) AS DateOfLastContact, 
    COUNT(DISTINCT CASE WHEN ContactAfterLTSA = 1 and PrimaryActivity = 1 THEN CONCAT(intr.InteractionTypeCode, intr.InteractionDate) ELSE NULL END ) AS TotalCountOfContacts, 
    COUNT(DISTINCT CASE WHEN ContactAfterLTSA = 1 AND GuarantorAttemptEver = 1  THEN intr.CDI ELSE NULL END ) AS DistinctCountContactedGuarantors 
FROM ActivityAttemptsDialer intr 
GROUP BY ExposureKey
""")
ActivityAttemptsDialerGroupped.createOrReplaceTempView("ActivityAttemptsDialerGroupped")

## Final Table

In [0]:
finaltbl1 = spark.sql(f"""
SELECT DISTINCT  CAST('{run_date}' AS DATE) as BusinessDate,
    de.PortfolioName AS PortfolioName,
    de.CDI , 
    de.ExposureKey,
    de.AssetSegment,
    de.AgreementKey,
    de.PartyTaxRegistrationNumber,
    de.WriteOffAmount,
    tbr.BalanceRange, 
    CAST(coalesce(iex.InitialAssignmentBalance, de.InitialAssignmentAmount) AS DECIMAL(18,2)) AS TLCAtMigrationDate,
    CAST(dea2.OutstandingTotalBalance AS DECIMAL(18,2)) AS PrevEoMTLC,
    CAST(de.OutstandingTotalBalance  AS DECIMAL(18,2)) AS CurrentTLC, 
    CAST(dea.OutstandingTotalBalance  AS DECIMAL(18,2)) AS TotalLegalClaimAtAppliedDate, 
    CAST(dea1.OutstandingTotalBalance  AS DECIMAL(18,2)) AS TotalLegalClaimAtApprovedDate, 
    CAST(de.TotalPrincipal AS DECIMAL(18,2)) AS PrincipalBalance, 
    CAST(de.TotalInterest AS DECIMAL(18,2)) AS Interest, 
    CAST(de.TotalExpense AS DECIMAL(18,2)) AS Expenses, 
    CAST(de.TotalAccruals AS DECIMAL(18,2)) AS Accrued,
    CAST(col.MonthlyPayments  AS DECIMAL(18,2)) MonthlyPayments, 
    CAST(col.TotalPaymentsAfterLTSA AS DECIMAL(18,2)) TotalPaymentsAfterAssignment,
    CAST(col.TotalPaymentsBeforeLTSA AS DECIMAL(18,2)) TotalPaymentsBeforeAssignment,  
    col.LastPaymentDate ,
    de.ExposureStatus,
    CASE WHEN sttl.HasActiveSettlementFlag IS TRUE THEN sttl.SettlementBucket ELSE de.Bucket END AS CurrentMonthBucket,
    mcb.MaxBucket as MaxContractBucket,
    CASE WHEN de.IsActiveFlag IS FALSE THEN 'Yes' ELSE 'No' END AS ClosedAccountFlag,
    CAST(de.ClosedDate AS DATE) AS ClosedAccountsDate,
    assgn.AssignedChannel, 
    prt.Segment,
    dpt.WorkableFlag AS WorkableFlag,
    dpt.NonWorkableReason AS NonWorkableReason,
    prt.subsegment AS Leverage,
    prt.ExtraSegment, 
    de.CoCStatus,
    CASE WHEN de.DenouncementIndicator IS TRUE THEN 'Yes' ELSE 'No' END AS SystemicDenounced,
    CASE WHEN de.LPSEng IS NOT NULL THEN 'Yes' ELSE 'No' END AS CaseUnderLegal,
    de.LPSEng AS LastLPS, 
    CAST(de.LPSDate AS DATE) AS LastLPSDate,
    CASE WHEN prod.ProductKey is not null THEN 'Yes' ELSE 'No' END AS AutoLoanFlag,    
    CASE WHEN collm.ExposureKey is not null then 'Y' else 'N' end AS RECollateralExists,
    coll.MinCollateralLien as RECollateralLien,
    CAST(coll.CollateralTotalAmount AS DECIMAL(18,2)) AS RECollateralAmount,
    CAST(coll.MinCollateralLienTotalAmount AS DECIMAL(18,2)) AS MinCollateralLienTotalAmount,
    CAST(sttl.DueAmount AS DECIMAL(18,2)) AS DueAmount,
    CAST(sttl.KeptRate AS DECIMAL(18,6)) AS KeptRate,
    sttl.ImplementationDateEver as ImplementationDateEver,
    sttl.SettlementCDI,
    CAST(sttl.CountofImplementationsPerAccountBeforeLTSA AS INT) CountofImplementationsPerAccountBeforeLTSA, 
    CAST(sttl.CountofImplementationsPerAccountAfterLTSA AS INT) CountofImplementationsPerAccount, 
    CAST(sttl.AppliedDate AS DATE),
    sttl.ApprovalDate,
    CAST(sttl.SettlementAmount2 AS DECIMAL(18,2)) SettlementAmount,
    CAST(sttl.HaircutAmount AS DECIMAL(18,2)) HaircutAmount, 
    CAST(NULL AS DECIMAL(18,6)) AS HaircutPercentage, -- CAST(sttl.HaircutPercentage AS DECIMAL(18,6)) HaircutPercentage, 
    CAST(sttl.HaircutPrincipalPercentage AS DECIMAL(18,6))HaircutPercPrincipalBalance, 
    CAST(sttl.TotalAmountDueAfterHaircut AS DECIMAL(18,2)) TotalAmountDueAfterHaircut,
    sttl.SettlementType, 
    sttl.SettlementTypeGroup,
    sttl.ApplicationType,
    sttl.Status as ModificationStatus, 
    IFNULL(sttl.SettlementFromMigration, 0) as MigratedSettlementFlag,
    sttl.NumberOfInstallments AS CountOfInstallments,
    sttl.NumberOfPastInstallments as CountOfPastInstalments, 
    CAST(sttl.NumberOfMissedInstallments AS DECIMAL(18,2)) AS CountofMissedIinstallmentPayments,
    sttl.ShortTerm_LongTermFlag as SettlementDuration, 
    sttl.FirstInstallmentDate2 as FirstInstalmentDate, 
    cast(sttl.MonthlyInstallmentAmount AS DECIMAL (18,2)) MonthlyInstallmentAmount,
    sttl.DownPaymentFlagForApplied,
    CAST(sttl.FirstInstallmentDateAfterApplication AS DATE),   
    sttl.ScheduledAmount,
    coalesce(prevEoML3869.InLawFlag, 0) as L3869PrevEoMFlag,
    coalesce(l3869.InLawFlag, 0) as L3869Flag,
    CASE WHEN l3869.InLawFlag = 1 THEN 'Under L3869'
        WHEN l3869.InLawFlag = 0 AND lawhist.L3869PrevFlag = 1 THEN 'Exit L3869'
        ELSE 'Not Under L3869' 
        END AS L3869StatusHistory,
    law.L3869Status1 AS L3869Status,
    law.L3869MonthExit as MonthExitFromL3869,
    law.L3869CourtDecision,
CASE
    WHEN law.L3869CourtDecision = 'ΔΕΚΤΗ ΑΠΟΦΑΣΗ' THEN 'Court Approval'
    WHEN law.L3869CourtDecision IN ('ΜΑΤΑΙΩΣΗ', 'ΜΑΤΑΙΩΣΗ ΛΟΓΩ COVID-19') THEN 'Paused'
    WHEN law.L3869CourtDecision IN ('ΑΝΑΜΟΝΗ ΑΠΟΤΕΛΕΣΜΑΤΟΣ', 'ΕΚΔΙΚΑΣΗ-ΑΝΑΜΟΝΗ ΑΠΟΦΑΣΗΣ') THEN 'Court Decision Pending'
    WHEN law.L3869CourtDecision = 'ΜΕΛΛΟΝΤΙΚΗ ΔΙΚΑΣΙΜΟΣ' THEN 'Court Hearing Pending'
    WHEN law.L3869CourtDecision = 'ΑΠΟΡΡΙΠΤΙΚΗ ΑΠΟΦΑΣΗ' THEN 'Court Rejection'
    WHEN law.L3869CourtDecision = 'ΑΠΑΡΑΔΕΚΤΗ ΣΥΖΗΤΗΣΗ' THEN 'Court Rejection'
    WHEN law.L3869CourtDecision = 'ΠΑΡΑΙΤΗΣΗ' THEN 'Resignations'
    WHEN law.L3869CourtDecision = 'ΥΛΟΠΟΙΗΣΗ ΣΥΜΒΑΣΗΣ ΡΥΘΜΙΣΗΣ -ΔΙΑΤΡΑΠΕΖΙΚΗ' THEN 'Settlement'
    WHEN law.L3869CourtDecision = 'ΕΚΠΤΩΣΗ' THEN 'Annulment'
    WHEN law.L3869CourtDecision = 'ΑΠΑΛΛΑΓΗ' THEN 'Paid off Court Decision'
    WHEN law.L3869CourtDecision = 'ΑΠΑΡΑΔΕΚΤΗ' THEN 'Rejected'
    WHEN law.L3869CourtDecision = 'ΔΙΚΑΣΤΙΚΗ ΕΞΟΦΛΗΣΗ' THEN 'Paid off Court Settlement'
    WHEN law.L3869CourtDecision IN ('ΜΗΔΕΠΟΤΕ ΑΣΚΗΘΕΙΣΑ', 'ΑΓΝΩΣΤΗ ΕΚΒΑΣΗ') THEN 'N/A'
    WHEN law.L3869CourtDecision = 'Δεκτή Δικαστική Απόφαση' THEN 'Court Approval'
    WHEN law.L3869CourtDecision = 'ΔΕΚΤΗ' THEN 'Court Approval'
    WHEN law.L3869CourtDecision = 'ΚΑΤΑΡΓΗΣΗ' THEN 'Cancellation'
    WHEN law.L3869CourtDecision = 'ΑΠΟΣΥΡΣΗ' THEN 'Withdrawal'
    WHEN law.L3869CourtDecision = 'ΣΥΜΒΙΒΑΣΤΙΚΗ ΕΞΟΦΛΗΣΗ' THEN 'Extra Judicial Settlement'
    WHEN law.L3869CourtDecision = 'ΠΑΡΑΙΤΗΣΗ ΜΗ ΕΜΠΡΟΘΕΣΜΗ ΚΑΤΑΘΕΣΗ Ν4745' THEN 'Cancellation 4745'
    ELSE law.L3869CourtDecision
END AS L3869CourtOutcomeMapping,
    law.L3869FinalCourtDate,
    law.PreviousSegmentation as PreviousSegmentation,
    law.L3869SubmissionDate as L3869ApplicationDate,
    law.L3869ResignationDate,
    coalesce(law.EkptosiFlag, 'No') as ExpulsionFlag,
    law.EkptosiStatus as ExpulsionStatus,  
    CAST(E.Total_Balance_ML AS STRING) AS `L3869MLAmount`,
    CASE WHEN E.Has_ML IS NOT NULL THEN E.Has_ML ELSE 'No' END AS MLinABExist, 
    CAST( CASE WHEN de.ExposureStatus NOT IN ('Define Delay', 'Writen Off')
    THEN de.DuePrincipal + de.DueInterest + de.PenaltyInterest + de.DueExpense + de.DueUnrealizedInterest + de.UnrealizedInterest +
    de.PenaltyUnrealizedInterest
    ELSE de.OutstandingTotalBalance END AS DECIMAL (18,2)) AS L3869TotalDebt, 
    ifnull(CASE WHEN de.IsActiveFlag = FALSE THEN NULL ELSE de.DaysPastDueFinal END, 0) as DPD, 
    CAST(de.OnBalance AS DECIMAL (18,2)) OnBalance,
    CASE WHEN lf.LetterFlagCDI IS NOT NULL THEN 'Yes' ELSE 'No' END AS DemandLetterFlag,
    CASE WHEN DueUnrealizedinterest + PenaltyUnrealizedinterest + Unrealizedinterest >0  then 'Yes'
                                                                                         ELSE 'No'
        END AS OffbalancesheetInterestFlag,
    ro.AmountOfHaircutLastRejectedOffer as HaircutAmountLastRejectedOffer,
    ro.HaircutPercentOfGBVLastRejectedOffer as HaircutPercentageLastRejectedOffer,
    ro.HaircutPercentPrincipalBalanceLastRejectedOffer as HaircutPercPrincipalBalanceLastRejectedOffer, 
    ro.CountOfInstallmentsLastRejectedOffer,
    ro.MonthlyInstallmentAmountLastRejectedOffer as InstallmentAmountLastRejectedOffer, 
    cc.StrategyStep, 
    cc.StrategyType, 
    CASE
        WHEN cc.QueueTrimmed IN (
            '_Profits_Settlement_Αρση_Εγγ_Settl_Buc_01-03',
            '_Collection_Profits_Settlements_DNC_ext',
            '_Collection_Profits_Settlements_ext',
            '_Collection_Qualco_Settlements_ext',
            '_OCW_Collection_Assigned_ext',
            '_Collection_Assign_to_Ext',
            '_B4+_from_Collection_to_Offer_Assign_int',
            '_B4+_Profits_Settlement_to_Offer_Assign_int',
            '_B4+_Qualco_Settlement_to_Offer_Assign_int',
            '_Offer_CREV_50-100%_CA_int',
            '_Offer_CREV_0-50%_CA_int',
            '_Offer_Assign_ext',
            '_Offer_FA_int',
            '_Offer_FA_no_officer_int',
            '_Offer_CA_Personal_Ass_int',
            '_Offer_CA_int',
            '_No_Segment_(Balance <50)',
            '_Default'
        ) THEN TRUE

        WHEN cc.QueueTrimmed = '_Slave_Accounts' AND ProcessType IN ('Collection_Settlement', 'Collection_Settlement_Paladino','Collection_Settlement_Teleperformance', 'Offer') THEN TRUE

        WHEN cc.QueueTrimmed IN (
        ' Vulnerable Debtor_Certification_',
        ' Vulnerable Debtor_Application_',
        '_Arotron_Legal_Phase_cases',
        '_Slave_Accounts',
        '_L3869_Accepted_SR',
        '_Profits_Settlement_almost_Paid_Off',
        '_Qualco_Settlement_almost_Paid_Off',
        '_Profits_Fulfilled_Αρση_Εγγύησης',
        '_Profits_Fulfilled_Settlement',
        '_Qualco_Fulfilled_Settlements',
        '_L3869',
        '_Profits_Settlement_Pending_Implementation',
        '_Profits_Settlement_Αρση_Εγγύησης_Settl_Buc00',
        '_Profits_Settlement_Bucket_00_DNC',
        '_Qualco_Settlement_Bucket_00_DNC',
        '_Profits_Settlement_Grace<10€_DNC',
        '_Profits_Settlement_under_11_Dpds_DNC',
        '_Profits_Settlement_Bucket_00',
        '_Qualco_Settlement_Bucket_00',
        '_Profits_Settlement_Grace<10€',
        '_Profits_Settlement_under_11_Dpds',
        '_Current_Bucket_00',
        '_Grace<10€',
        '_Under_11_Dpds',
        '_SR_Non-Delinquent_Profits_Settlements',
        '_SR_Non-Delinquent_Qualco_Settlements',
        '_Bankruptcy_queue_non_final_stages',
        '_Bankruptcy_final_stage_completion',
        '_OCW_queue_non_final_stages',
        '_OCW_final_stage_completion',
        '_No_Treatment_state_guaranteed'
    ) THEN FALSE

        ELSE NULL
    END AS ActiveOrPassiveAssignment,

    -- Extra fields used in summaries export
    sttl.isActiveSettlement,
    CASE WHEN sttl.Status <> 'No Settlement/Restructure' THEN 1 ELSE 0 END AS ImplementationFlag, 
    CASE WHEN sttl.ApprovalDate IS NOT NULL THEN 1 ELSE 0 END AS ApprovalFlag,
    CASE WHEN sttl.AppliedDate IS NOT NULL THEN 1 ELSE 0 END AS AppliedFlag
FROM dexposure de
LEFT JOIN initializationexposureinfo iex 
on de.Exposurekey = iex.Exposurekey and de.PortfolioName = iex.PortfolioName
LEFT JOIN TaxBalanceRange tbr
ON de.CDI = tbr.CDI
LEFT JOIN Collections col
ON de.ExposureKey = col.ExposureKey
LEFT JOIN casecode cc 
on cc.CaseCodeKey = de.CaseCode
LEFT JOIN MaxContractBucket mcb 
ON de.AgreementKey = mcb.AgreementKey
LEFT JOIN assignment assgn
ON regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(assgn.ExposureKey, '[^0-9]', '')
AND regexp_replace(de.CDI, '[^0-9]', '') = regexp_replace(assgn.CDI, '[^0-9]', '') 
AND de.PortfolioName = assgn.PortfolioName
-- LEFT JOIN dparty dp
-- ON de.CDI = dp.CDI
LEFT JOIN party prt
ON de.CDI = prt.CDI
AND de.PortfolioName = prt.PortfolioName
LEFT JOIN CollateralMappingTempLien coll
ON de.ExposureKey = coll.ExposureKey
LEFT JOIN 
(select distinct ExposureKey 
    from CollateralMapping 
    where RealEstateCollateralStatus = 'Active' 
) collm
on collm.ExposureKey = de.ExposureKey
LEFT JOIN settlement_fields sttl
ON de.ExposureKey = sttl.ExposureKey
LEFT JOIN dunderlaw law
ON de.CDI = law.CDI
LEFT JOIN dunderlaw_history lawhist
ON de.CDI = lawhist.CDI
LEFT JOIN isPrevEoML3869 PrevEoMl3869
ON de.CDI = PrevEoMl3869.CDI    
LEFT JOIN isCurrentL3869 l3869
ON de.CDI = l3869.CDI                    
LEFT JOIN `{env}-reportinputfiles`.investor.mlexist E 
ON regexp_replace(de.CDI, '[^0-9]', '') = regexp_replace(E.CIF, '[^0-9]', '')
AND de.PortfolioName = E.Portfolio
LEFT JOIN letterflagexist lf
ON de.CDI = lf.LetterFlagCDI
LEFT JOIN DPortfolioToolThirdParties dpt
ON dpt.exposurekey=de.exposurekey
LEFT JOIN dexposure_all dea 
ON dea.ExposureKey = sttl.Exposurekey and dea.BusinessDate = sttl.AppliedDate
LEFT JOIN dexposure_all dea1  
ON dea1.ExposureKey = sttl.Exposurekey and dea1.BusinessDate = sttl.ApprovalDate  
LEFT JOIN dexposure_all dea2   
ON dea2.ExposureKey = de.Exposurekey and dea2.BusinessDate = '{PreviousMonthLastWorkingDate}'
LEFT JOIN rej_off ro 
ON de.AgreementKey = ro.AgreementKey 
LEFT JOIN product prod
on de.ProductKey = prod.ProductKey 
""")
 
finaltbl1.createOrReplaceTempView("finaltbl1")

In [0]:
finaltbl = spark.sql(f"""
SELECT DISTINCT A.*,                                
        B.EAP_Status as EAPStatus,
        cast(B.Last_EAP_Date as date) as LastEAPDate,
        B.EAPDateAfterMigration as EAPDateAfterAssignmentFlag,       
        Coalesce(re.Positive_RE_Exists,'No') AS PositiveREExist ,       
        AG.TotalCountOfOffersRejected,    
        AG.DateOfLastRejectedOffer,            
        ARNegative.InteractionType as RejectionReasons, 
        DG.DateofFirstAttemptAfterLTSA as FirstAttemptedDate, 
        DG.DateOfLastAttemptAfterLTSA LastAttemptedDate, 
        DG.TotalCountOfAttempts_after_LTSA CountOfAttempts, 
        DG.DistinctAttemptDays_after_LTSA as CountOfDistinctAttemptedDays,
        LTSA.LastAttemptDateBeforeLTSA ,
        DG.TotalCountOfGuarantorAttempts as CountOfAttemptsGuarantor,
        DG.DateOfLastGuarantorAttempt as LastAttemptDateGuarantor,
        IFNULL(DG.DistinctCountAttemptedGuarantors, 0) AS DistinctAttemptedGuarantors,
        DG.DateOfFirstContact as FirstContactDate,
        DG.DateOfLastContact as LastContactDate,
        IFNULL(DG.DistinctCountContactedGuarantors, 0) AS DistinctContactedGuarantors,
        AG.TotalCountOfRPCs as CountOfRPC,
        ARRPC.AttemptOutcome as LastRPCStatus,
        AG.DateOfLastRPC as LastRPCStatusDate,
        AG.TotalCountOfGuarantorRPC as CountOfRPCGuarantor,
        AG.DateOfLastGuarantorRPC as LastRPCDateGuarantor,
        AG.PositiveRPC as CountOfRPCPositive,
        AG.NegativeRPC as CountOfRPCNegative,
        LTSA.LastRPCDateBeforeLTSA,
        AG.LastInboundDate as LastInboundActivityDate,    
        LTSA.AttemptBeforeLTSA,
        LTSA.RPCBeforeLTSA, 
        LTSA.RPCGuarantorBeforeLTSA,
        LTSA.LastRPCStatusBeforeLTSA,

        -- Extra fields used in summaries export
        CASE WHEN DG.TotalCountOfAttempts_after_LTSA > 0 THEN 1 ELSE 0 END AS WorkedFlag,
        CASE WHEN AG.TotalCountOfRPCs > 0 THEN 1 ELSE 0 END AS ContactedFlag
                               
FROM finaltbl1 A
LEFT JOIN eap B
ON A.AgreementKey = B.AgreementKey
LEFT JOIN ltsa on ltsa.BorrowerCode = A.CDI and ltsa.loanCode = A.ExposureKey and ltsa.rnk = 1
LEFT JOIN ActivitiesGroupped AS AG
ON A.ExposureKey = AG.ExposureKey
LEFT JOIN ActivityAttemptsDialerGroupped AS DG
ON A.ExposureKey = DG.ExposureKey
LEFT JOIN ActivitiesRanked AS ARNegative
ON A.ExposureKey = ARNegative.ExposureKey and ARNegative.NegativeAfterCutOffRank = 1
LEFT JOIN ActivitiesRanked AS ARRPC
ON A.ExposureKey = ARRPC.ExposureKey and ARRPC.RPCRank = 1
LEFT JOIN re
on A.AgreementKey = re.AgreementKey and re.rank = 1
                    
""")
finaltbl.createOrReplaceTempView("finaltbl")

In [0]:
Final = spark.sql(
f'''
SELECT
    CAST(BusinessDate AS DATE) AS BusinessDate,
    CAST(PortfolioName AS STRING) AS PortfolioName,
    CAST(CDI AS STRING) AS CDI,
    CAST(ExposureKey AS STRING) AS ExposureKey,
    CAST(AssetSegment AS STRING) AS AssetSegment,
    CAST(AgreementKey AS STRING) AS AgreementKey,
    CAST(PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber,
    CAST(ifnull(WriteOffAmount, 0) AS DECIMAL(18,2)) AS WriteOffAmount,
    CAST(BalanceRange AS STRING) AS BalanceRange,
    CAST(ifnull(TLCAtMigrationDate, 0) AS DECIMAL(18,2)) AS TLCAtMigrationDate,
    CAST(ifnull(PrevEoMTLC, 0) AS DECIMAL(18,2)) AS PrevEoMTLC,
    CAST(ifnull(CurrentTLC, 0) AS DECIMAL(18,2)) AS CurrentTLC,
    CAST(ifnull(TotalLegalClaimAtAppliedDate, 0) AS DECIMAL(18,2)) AS TotalLegalClaimAtAppliedDate,
    CAST(ifnull(TotalLegalClaimAtApprovedDate, 0) AS DECIMAL(18,2)) AS TotalLegalClaimAtApprovedDate,
    CAST(ifnull(PrincipalBalance, 0) AS DECIMAL(18,2)) AS PrincipalBalance,
    CAST(ifnull(Interest, 0) AS DECIMAL(18,2)) AS Interest,
    CAST(ifnull(Expenses, 0) AS DECIMAL(18,2)) AS Expenses,
    CAST(ifnull(Accrued, 0) AS DECIMAL(18,2)) AS Accrued,
    CAST(ifnull(MonthlyPayments, 0) AS DECIMAL(18,2)) AS MonthlyPayments,
    CAST(ifnull(TotalPaymentsAfterAssignment, 0) AS DECIMAL(18,2)) AS TotalPaymentsAfterAssignment,
    CAST(ifnull(TotalPaymentsBeforeAssignment, 0) AS DECIMAL(18,2)) AS TotalPaymentsBeforeAssignment,
    CAST(LastPaymentDate AS DATE) AS LastPaymentDate,
    CAST(ExposureStatus AS STRING) AS ExposureStatus,
    CAST(CurrentMonthBucket AS INT) AS CurrentMonthBucket,
    CAST(MaxContractBucket AS INT) AS MaxContractBucket,
    CAST(ClosedAccountFlag AS STRING) AS ClosedAccountFlag,
    CAST(ClosedAccountsDate AS DATE) AS ClosedAccountsDate,
    CAST(AssignedChannel AS STRING) AS AssignedChannel,
    CAST(Segment AS STRING) AS Segment,
    CAST(WorkableFlag AS BOOLEAN) AS WorkableFlag,
    CAST(NonWorkableReason AS STRING) AS NonWorkableReason,
    CAST(Leverage AS STRING) AS Leverage,
    CAST(ExtraSegment AS STRING) AS ExtraSegment,
    CAST(CoCStatus AS STRING) AS CoCStatus,
    CAST(SystemicDenounced AS STRING) AS SystemicDenounced,
    CAST(CaseUnderLegal AS STRING) AS CaseUnderLegal,
    CAST(LastLPS AS STRING) AS LastLPS,
    CAST(LastLPSDate AS DATE) AS LastLPSDate,
    CAST(AutoLoanFlag AS STRING) AS AutoLoanFlag,
    CAST(RECollateralExists AS STRING) AS RECollateralExists,
    CAST(RECollateralLien AS INT) AS RECollateralLien,
    CAST(ifnull(RECollateralAmount, 0) AS DECIMAL(18,2)) AS RECollateralAmount,
    CAST(ifnull(MinCollateralLienTotalAmount, 0) AS DECIMAL(18,2)) AS MinCollateralLienTotalAmount,
    CAST(ifnull(DueAmount, 0) AS DECIMAL(18,2)) AS DueAmount,
    CAST(KeptRate AS DECIMAL(18,6)) AS KeptRate,
    CAST(ImplementationDateEver AS DATE) AS ImplementationDateEver,
    CAST(SettlementCDI AS STRING) AS SettlementCDI,
    CAST(ifnull(CountofImplementationsPerAccountBeforeLTSA, 0) AS INT) AS CountofImplementationsPerAccountBeforeLTSA,
    CAST(ifnull(CountofImplementationsPerAccount, 0) AS INT) AS CountofImplementationsPerAccount,
    CAST(AppliedDate AS DATE) AS AppliedDate,
    CAST(ApprovalDate AS DATE) AS ApprovalDate,
    CAST(SettlementAmount AS DECIMAL(18,2)) AS SettlementAmount,
    CAST(HaircutAmount AS DECIMAL(18,2)) AS HaircutAmount,
    CAST(HaircutPercentage AS DECIMAL(18,6)) AS HaircutPercentage,
    CAST(HaircutPercPrincipalBalance AS DECIMAL(18,6)) AS HaircutPercPrincipalBalance,
    CAST(TotalAmountDueAfterHaircut AS DECIMAL(18,2)) AS TotalAmountDueAfterHaircut,
    CAST(SettlementType AS STRING) AS SettlementType,
    CAST(SettlementTypeGroup AS STRING) AS SettlementTypeGroup,
    CAST(ApplicationType AS STRING) AS ApplicationType,
    CAST(ModificationStatus AS STRING) AS ModificationStatus,
    CAST(MigratedSettlementFlag AS INT) AS MigratedSettlementFlag,
    CAST(ifnull(CountOfInstallments, 0) AS INT) AS CountOfInstallments,
    CAST(ifnull(CountOfPastInstalments, 0) AS INT) AS CountOfPastInstalments,
    CAST(ifnull(CountofMissedIinstallmentPayments, 0) AS INT) AS CountofMissedIinstallmentPayments,
    CAST(SettlementDuration AS STRING) AS SettlementDuration,
    CAST(FirstInstalmentDate AS DATE) AS FirstInstalmentDate,
    CAST(ifnull(MonthlyInstallmentAmount, 0) AS DECIMAL(18,2)) AS MonthlyInstallmentAmount,
    CAST(DownPaymentFlagForApplied AS BOOLEAN) AS DownPaymentFlagForApplied,
    CAST(FirstInstallmentDateAfterApplication AS DATE) AS FirstInstallmentDateAfterApplication,
    CAST(ScheduledAmount AS DECIMAL(18,2)) AS ScheduledAmount,
    CAST(L3869PrevEoMFlag AS INT) AS L3869PrevEoMFlag,
    CAST(L3869Flag AS INT) AS L3869Flag,
    CAST(L3869StatusHistory AS STRING) AS L3869StatusHistory,
    CAST(L3869Status AS STRING) AS L3869Status,
    CAST(MonthExitFromL3869 AS STRING) AS MonthExitFromL3869,
    CAST(L3869CourtDecision AS STRING) AS L3869CourtDecision,
    CAST(L3869CourtOutcomeMapping AS STRING) AS L3869CourtOutcomeMapping,
    CAST(L3869FinalCourtDate AS DATE) AS L3869FinalCourtDate,
    CAST(PreviousSegmentation AS STRING) AS PreviousSegmentation,
    CAST(L3869ApplicationDate AS DATE) AS L3869ApplicationDate,
    CAST(L3869ResignationDate AS DATE) AS L3869ResignationDate,
    CAST(ExpulsionFlag AS STRING) AS ExpulsionFlag,
    CAST(ExpulsionStatus AS STRING) AS ExpulsionStatus,
    CAST(L3869MLAmount AS STRING) AS L3869MLAmount,
    CAST(MLinABExist AS STRING) AS MLinABExist,
    CAST(L3869TotalDebt AS DECIMAL(18,2)) AS L3869TotalDebt,
    CAST(DPD AS INT) AS DPD,
    CAST(ifnull(OnBalance, 0) AS DECIMAL(18,2)) AS OnBalance,
    CAST(DemandLetterFlag AS STRING) AS DemandLetterFlag,
    CAST(OffbalancesheetInterestFlag AS STRING) AS OffbalancesheetInterestFlag,
    CAST(HaircutAmountLastRejectedOffer AS DECIMAL(18,2)) AS HaircutAmountLastRejectedOffer,
    CAST(HaircutPercentageLastRejectedOffer AS DECIMAL(18,6)) AS HaircutPercentageLastRejectedOffer,
    CAST(HaircutPercPrincipalBalanceLastRejectedOffer AS DECIMAL(18,6)) AS HaircutPercPrincipalBalanceLastRejectedOffer,
    CAST(CountOfInstallmentsLastRejectedOffer AS INT) AS CountOfInstallmentsLastRejectedOffer,
    CAST(InstallmentAmountLastRejectedOffer AS DECIMAL(18,6)) AS InstallmentAmountLastRejectedOffer,
    CAST(StrategyStep AS STRING) AS StrategyStep,
    CAST(StrategyType AS STRING) AS StrategyType,
    CAST(isActiveSettlement AS BOOLEAN) AS isActiveSettlement,
    CAST(ImplementationFlag AS INT) AS ImplementationFlag,
    CAST(ApprovalFlag AS INT) AS ApprovalFlag,
    CAST(AppliedFlag AS INT) AS AppliedFlag,
    CAST(EAPStatus AS STRING) AS EAPStatus,
    CAST(LastEAPDate AS DATE) AS LastEAPDate,
    CAST(EAPDateAfterAssignmentFlag AS STRING) AS EAPDateAfterAssignmentFlag,
    CAST(PositiveREExist AS STRING) AS PositiveREExist,
    CAST(ifnull(TotalCountOfOffersRejected, 0) AS INT) AS TotalCountOfOffersRejected,
    CAST(DateOfLastRejectedOffer AS DATE) AS DateOfLastRejectedOffer,
    CAST(RejectionReasons AS STRING) AS RejectionReasons,
    CAST(FirstAttemptedDate AS DATE) AS FirstAttemptedDate,
    CAST(LastAttemptedDate AS DATE) AS LastAttemptedDate,
    CAST(ifnull(CountOfAttempts, 0) AS INT) AS CountOfAttempts,
    CAST(ifnull(CountOfDistinctAttemptedDays, 0) AS INT) AS CountOfDistinctAttemptedDays,
    CAST(LastAttemptDateBeforeLTSA AS DATE) AS LastAttemptDateBeforeLTSA,
    CAST(ifnull(CountOfAttemptsGuarantor, 0) AS INT) AS CountOfAttemptsGuarantor,
    CAST(LastAttemptDateGuarantor AS DATE) AS LastAttemptDateGuarantor,
    CAST(ifnull(DistinctAttemptedGuarantors, 0) AS INT) AS DistinctAttemptedGuarantors,
    CAST(FirstContactDate AS DATE) AS FirstContactDate,
    CAST(LastContactDate AS DATE) AS LastContactDate,
    CAST(ifnull(DistinctContactedGuarantors, 0) AS INT) AS DistinctContactedGuarantors,
    CAST(ifnull(CountOfRPC, 0) AS INT) AS CountOfRPC,
    CAST(LastRPCStatus AS STRING) AS LastRPCStatus,
    CAST(LastRPCStatusDate AS DATE) AS LastRPCStatusDate,
    CAST(ifnull(CountOfRPCGuarantor, 0) AS INT) AS CountOfRPCGuarantor,
    CAST(LastRPCDateGuarantor AS DATE) AS LastRPCDateGuarantor,
    CAST(ifnull(CountOfRPCPositive, 0) AS INT) AS CountOfRPCPositive,
    CAST(ifnull(CountOfRPCNegative, 0) AS INT) AS CountOfRPCNegative,
    CAST(LastRPCDateBeforeLTSA AS DATE) AS LastRPCDateBeforeLTSA,
    CAST(LastInboundActivityDate AS DATE) AS LastInboundActivityDate,
    CAST(AttemptBeforeLTSA AS INT) AS AttemptBeforeLTSA,
    CAST(RPCBeforeLTSA AS INT) AS RPCBeforeLTSA,
    CAST(RPCGuarantorBeforeLTSA AS INT) AS RPCGuarantorBeforeLTSA,
    CAST(LastRPCStatusBeforeLTSA AS STRING) AS LastRPCStatusBeforeLTSA,
    CAST(WorkedFlag AS INT) AS WorkedFlag,
    CAST(ContactedFlag AS INT) AS ContactedFlag, 
    CAST(ActiveOrPassiveAssignment AS STRING) AS ActiveOrPassiveAssignment
FROM finaltbl;

''')
Final.createOrReplaceTempView("Final")

In [0]:
target_table_path_linebyline = f"`{env}-cdh-delivery`.investorreports.hoist_linebyline"
spark.sql(f"""DELETE FROM {target_table_path_linebyline} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_linebyline) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_linebyline} SYNC METADATA")